# Visión e imágenes

Redes convolucionales auto-decorrelacionadas, reconstrucción con tripletas, GAN, video y un clasificador de imágenes.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

> **Medido en el banco** (`laboratorio/banco/`): ver la sección «cuaderno 04» de `laboratorio/banco/README.md`. Las capas se pasaron de TensorFlow a PyTorch para medirlas igual que las demás.

## Red convolucional auto-decorrelada (AODC-NN)

Cada convolución castiga que sus mapas se parezcan entre sí. En el banco: arquitectura `cnn_decorrelada` (imágenes) y pieza `decorrelacion` (capa densa).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt # Para visualización opcional

# --- 1. Función de Pérdida para la Decorrelación de Características de UNA Capa ---

def decorrelation_loss_intra_layer(feature_maps, lambda_coeff=0.1):
    """
    Calcula la pérdida para penalizar la redundancia/correlación entre
    los mapas de características (canales/filtros) de una sola capa convolucional.
    """
    # Use tf.shape for graph compatibility
    shape = tf.shape(feature_maps)
    batch_size = shape[0]
    height_width = shape[1] * shape[2]
    filters = shape[3]

    # Use tf.cond for conditional logic based on tensors
    def zero_loss():
        return tf.constant(0.0, dtype=tf.float32)

    def calculate_loss():
        # Flatten spatial dimensions
        flattened_features = tf.reshape(feature_maps, (batch_size, height_width, filters))

        # Handle case where height_width is <= 1
        def zero_loss_small_spatial():
            return tf.constant(0.0, dtype=tf.float32)

        def calculate_covariance_loss():
            # Centrar los datos por filtro
            mean_per_filter = tf.reduce_mean(flattened_features, axis=1, keepdims=True)
            centered_features = flattened_features - mean_per_filter  # (batch_size, H*W, filters)

            # Calculate the covariance matrix per batch
            # Avoid division by zero if (H*W - 1) is 0 or negative
            divisor = tf.cast(height_width, tf.float32) - 1.0
            # Use tf.where or tf.math.maximum to handle the divisor <= 0 case
            safe_divisor = tf.where(divisor <= 0, tf.constant(1.0, dtype=tf.float32), divisor) # Use 1.0 if divisor <= 0

            covariance_matrix = tf.matmul(centered_features, centered_features, transpose_a=True) / safe_divisor

            # Queremos que los elementos fuera de la diagonal sean pequeños.
            identity = tf.eye(filters, dtype=tf.float32)
            off_diagonal_mask = 1.0 - identity

            off_diagonal_loss = tf.reduce_sum(tf.square(covariance_matrix * off_diagonal_mask), axis=[-1, -2])

            total_loss = tf.reduce_mean(off_diagonal_loss)

            # Ensure the loss is zero if the original divisor was <= 0
            return tf.where(divisor <= 0, tf.constant(0.0, dtype=tf.float32), lambda_coeff * total_loss)

        # Conditional calculation based on height_width
        return tf.cond(height_width <= 1, zero_loss_small_spatial, calculate_covariance_loss)


    # Conditional calculation based on the number of filters
    return tf.cond(filters <= 1, zero_loss, calculate_loss)


# --- 2. Capa Convolucional Personalizada con Auto-Decorrelación (sin cambios) ---

class AutoDecorrelatingConv2D(layers.Conv2D):
    """
    Una capa Conv2D que añade una pérdida de decorrelación entre sus propios
    mapas de características (filtros) para fomentar representaciones no redundantes.
    """
    def __init__(self, filters, kernel_size, lambda_redundancia=0.1, **kwargs):
        super(AutoDecorrelatingConv2D, self).__init__(filters, kernel_size, **kwargs)
        self.lambda_redundancia = lambda_redundancia

    def call(self, inputs):
        activations = super(AutoDecorrelatingConv2D, self).call(inputs)

        # Añade la pérdida de decorrelación. Esta línea es clave.
        # add_loss will work correctly with the tensor returned by decorrelation_loss_intra_layer
        self.add_loss(decorrelation_loss_intra_layer(activations, self.lambda_redundancia))

        return activations

# --- 3. Construcción del Modelo AODC-NN para MNIST ---

def construir_aodc_nn_mnist(input_shape, num_classes, lambda_redundancia_per_layer=0.001):
    """
    Construye una Red Neuronal Convolucional Auto-Orto-Diversa adaptada para MNIST.
    """
    input_layer = keras.Input(shape=input_shape, name="input_mnist_image")

    # Bloques convolucionales auto-decorrelantes
    x = AutoDecorrelatingConv2D(32, (3, 3), activation='relu', padding='same',
                                lambda_redundancia=lambda_redundancia_per_layer, name="aodc_conv1")(input_layer)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.BatchNormalization()(x) # Añadir BN para estabilidad en CNNs

    x = AutoDecorrelatingConv2D(64, (3, 3), activation='relu', padding='same',
                                lambda_redundancia=lambda_redundancia_per_layer, name="aodc_conv2")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.BatchNormalization()(x)

    x = AutoDecorrelatingConv2D(128, (3, 3), activation='relu', padding='same',
                                lambda_redundancia=lambda_redundancia_per_layer, name="aodc_conv3")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.BatchNormalization()(x)

    # Aplanar y capas densas para clasificación
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu', name="dense_features")(x)
    x = layers.Dropout(0.5)(x) # Dropout para regularización adicional
    output_layer = layers.Dense(num_classes, activation='softmax', name="output_prediction")(x)

    model = keras.Model(inputs=input_layer, outputs=output_layer, name="AODC_NN_MNIST")
    return model

# --- 4. Cargar y Preprocesar el dataset MNIST ---
print("--- Cargando y preprocesando el dataset MNIST ---")
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Las imágenes MNIST son (28, 28) en escala de grises. Keras espera (alto, ancho, canales).
# Escala de grises tiene 1 canal.
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

# Número de clases de MNIST (0-9)
NUM_CLASSES_MNIST = 10
# Forma de entrada para el modelo
INPUT_SHAPE_MNIST = (28, 28, 1)

# --- 5. Parámetros y Construcción del Modelo ---
LAMBDA_REDUNDANCIA_PER_LAYER_MNIST = 0.0001 # Ajusta este valor! Puede ser más bajo para MNIST.

print("\n--- Construyendo la Red Neuronal Convolucional Auto-Orto-Diversa (AODC-NN) para MNIST ---")
model_aodc_mnist = construir_aodc_nn_mnist(
    input_shape=INPUT_SHAPE_MNIST,
    num_classes=NUM_CLASSES_MNIST,
    lambda_redundancia_per_layer=LAMBDA_REDUNDANCIA_PER_LAYER_MNIST
)

model_aodc_mnist.summary()

# --- 6. Compilar el Modelo ---
print("\n--- Compilando el modelo para MNIST ---")
model_aodc_mnist.compile(optimizer='adam',
                         loss='sparse_categorical_crossentropy',
                         metrics=['accuracy'])

# --- 7. Entrenar el Modelo ---
print("\n--- Iniciando el entrenamiento del modelo AODC-NN para MNIST ---")
history_mnist = model_aodc_mnist.fit(x_train, y_train,
                                     epochs=10, # Unas pocas epochs para MNIST suelen ser suficientes
                                     batch_size=64,
                                     validation_split=0.1, # Usar una parte de training para validación
                                     verbose=1)

print("\n--- Entrenamiento finalizado ---")

# --- 8. Evaluar el Rendimiento Final ---
print("\n--- Evaluando el modelo en el conjunto de prueba de MNIST ---")
loss_mnist, accuracy_mnist = model_aodc_mnist.evaluate(x_test, y_test, verbose=1)
print(f"Pérdida en datos de prueba de MNIST: {loss_mnist:.4f}")
print(f"Precisión en datos de prueba de MNIST: {accuracy_mnist:.4f}")

# --- 9. Opcional: Visualizar las pérdidas y la precisión ---
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_mnist.history['loss'], label='Pérdida de Entrenamiento')
plt.plot(history_mnist.history['val_loss'], label='Pérdida de Validación')
plt.title('Pérdida durante el Entrenamiento')
plt.xlabel('Epoch')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_mnist.history['accuracy'], label='Precisión de Entrenamiento')
plt.plot(history_mnist.history['val_accuracy'], label='Precisión de Validación')
plt.title('Precisión durante el Entrenamiento')
plt.xlabel('Epoch')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# --- 10. Opcional: Inspeccionar la pérdida de decorrelación para una capa ---
# Para obtener los valores de la pérdida de decorrelación de las capas individuales:
# Keras añade las pérdidas internas a model.losses. Durante el entrenamiento, estas se suman.
# Para verlas individualmente para una muestra, tendrías que ejecutar la capa directamente:
print("\n--- Inspeccionando el valor de la pérdida de decorrelación para una capa (aodc_conv1) ---")
# Crear un modelo auxiliar para obtener solo la salida de la primera capa AODC_Conv2D
first_aodc_layer_output_model = keras.Model(inputs=model_aodc_mnist.input,
                                            outputs=model_aodc_mnist.get_layer("aodc_conv1").output)

# Obtener activaciones para una muestra de prueba
sample_input = x_test[0:1] # Primer ejemplo del set de prueba
sample_activations_conv1 = first_aodc_layer_output_model.predict(sample_input)

# Calcular la pérdida de decorrelación para estas activaciones
# Pass the sample_activations as a TensorFlow tensor for decorrelation_loss_intra_layer
calculated_decor_loss_sample = decorrelation_loss_intra_layer(
    tf.constant(sample_activations_conv1, dtype=tf.float32),
    lambda_coeff=LAMBDA_REDUNDANCIA_PER_LAYER_MNIST
)
print(f"Pérdida de decorrelación calculada para aodc_conv1 (una muestra): {calculated_decor_loss_sample.numpy():.6f}")
# Un valor bajo (cercano a 0) sugeriría que los filtros de esa capa han aprendido a ser menos redundantes.

## El «papelito preguntón»: extracción de características orto-diversas (ODFE-NN)

H1 extrae, H2 ve la entrada y a H1, H1 se reajusta viendo a H2, y se castiga que H1 y H2 se parezcan. Une las versiones RDCR-NN (re-extracción consciente) y NIDI-NN (que castigaba la covarianza en lugar del coseno). En el banco: `papelito_pregunton`, `papelito_covarianza` y `fusion_reciproca` (sin castigo).

In [ ]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# --- 1. Definición de las Capas de Extracción (H1 y H2) ---

class ExtractorCapaUno(layers.Layer):
    """
    Capa Oculta 1: Extrae características principales.
    Puede recibir contexto de H2 para ajuste.
    """
    def __init__(self, units, **kwargs):
        super(ExtractorCapaUno, self).__init__(**kwargs)
        self.units = units
        self.dense_from_input = layers.Dense(units, activation='relu', name="h1_from_input")
        # Esta capa procesa el contexto recibido de H2
        self.dense_from_h2_context = layers.Dense(units, activation='relu', name="h1_from_h2_context")
        self.combiner = layers.Concatenate(axis=-1, name="h1_combiner") # Para combinar si hay contexto
        self.final_h1_activation = layers.Dense(units, activation='relu', name="final_h1_activation")

    def call(self, inputs, h2_context=None):
        x_from_input = self.dense_from_input(inputs)

        if h2_context is not None:
            x_from_h2 = self.dense_from_h2_context(h2_context)
            combined = self.combiner([x_from_input, x_from_h2])
            return self.final_h1_activation(combined)
        else:
            return x_from_input

class ExtractorCapaDos(layers.Layer):
    """
    Capa Oculta 2: Extrae características complementarias/disjuntas.
    Recibe de la Entrada y de H1.
    """
    def __init__(self, units, **kwargs):
        super(ExtractorCapaDos, self).__init__(**kwargs)
        self.units = units
        self.dense_from_input = layers.Dense(units, activation='relu', name="h2_from_input")
        self.dense_from_h1 = layers.Dense(units, activation='relu', name="h2_from_h1")
        self.combiner = layers.Concatenate(axis=-1, name="h2_combiner")
        self.final_h2_activation = layers.Dense(units, activation='relu', name="final_h2_activation")

    def call(self, inputs_original, h1_output):
        # H2 siempre recibe la entrada original
        x_from_input = self.dense_from_input(inputs_original)
        # H2 también recibe y procesa la salida de H1
        x_from_h1 = self.dense_from_h1(h1_output)

        combined = self.combiner([x_from_input, x_from_h1])
        return self.final_h2_activation(combined)

# --- 2. Función de Pérdida Personalizada con Penalización de Redundancia ---

# Esta función se usará como un término adicional a la pérdida principal
def decorrelation_loss(h1_features, h2_features, lambda_coeff=0.1):
    """
    Penaliza la similitud de las características entre h1_features y h2_features.
    Cuanto más similares sean, mayor la pérdida.
    """
    # Aseguramos que ambas tienen la misma dimensión para el cálculo de similitud.
    # Si units_h1 != units_h2, se requeriría una capa Dense para proyectar a una dimensión común
    # o usar una penalización basada en covarianza/información mutua que no requiera same_dim.

    # Para simplicidad y que funcione con Dense:
    # Si las unidades de H1 y H2 son diferentes, se necesitaría una proyección antes de calcular la similitud.
    # Por ahora, asumimos que units_h1 == units_h2 para el cálculo de similitud directa,
    # o se proyectan a una dimensión común si se usan diferentes.

    # Normalizamos los vectores de características para calcular la similitud de coseno
    h1_norm = tf.linalg.normalize(h1_features, axis=-1)[0]
    h2_norm = tf.linalg.normalize(h2_features, axis=-1)[0]

    # Similitud de coseno (dot product de vectores normalizados)
    # tf.einsum('bi,bi->b', h1_norm, h2_norm) calcula el producto punto por batch
    similitud_coseno = tf.reduce_sum(h1_norm * h2_norm, axis=-1)

    # Queremos que la similitud sea pequeña, así que penalizamos los valores grandes.
    # Usamos el cuadrado para que siempre sea positivo y penalice más fuertemente.
    loss = tf.reduce_mean(tf.square(similitud_coseno))

    return lambda_coeff * loss

# --- 3. Construcción del Modelo ODFE-NN Completo ---

def construir_odfe_nn(input_shape, units_h1, units_h2, output_units, lambda_redundancia=0.1):
    entrada_modelo = keras.Input(shape=input_shape, name="input_layer")

    # Instanciar las capas de extracción
    extractor_h1 = ExtractorCapaUno(units=units_h1, name="extractor_h1")
    extractor_h2 = ExtractorCapaDos(units=units_h2, name="extractor_h2")

    # --- Flujo de Procesamiento para Simular la Interacción ("Papelito Preguntón") ---
    # Esto simula un par de rondas de "pregunta y ajuste" entre las capas.
    # En un modelo real más complejo, esto podría ser un RNN Cell personalizado.

    # Ronda 1: Extracción inicial
    # H1 extrae de la entrada (sin contexto de H2 aún)
    h1_features_t0 = extractor_h1(entrada_modelo, h2_context=None)
    # H2 extrae de la entrada y de las características iniciales de H1
    h2_features_t0 = extractor_h2(entrada_modelo, h1_features_t0)

    # Ronda 2: Refinamiento y Ajuste ("No la tienes, entonces me la quedo" o "Sí la tienes, ajústame")
    # H1 refina sus características usando el contexto de H2_t0
    # Esta es la "retroalimentación" para que H1 considere lo que H2 ya extrajo
    h1_features_t1 = extractor_h1(entrada_modelo, h2_context=h2_features_t0)
    # H2 refina sus características usando el H1_t1 ajustado
    # Esta es la "segunda pregunta" de H2 a H1 sobre las características refinadas
    h2_features_t1 = extractor_h2(entrada_modelo, h1_features_t1)

    # Las características finales después de la interacción
    final_h1_features = h1_features_t1
    final_h2_features = h2_features_t1

    # --- Combinación de las características finales para la tarea principal ---
    # Aquí puedes elegir cómo combinar: concatenar, sumar, etc.
    # Concatenar es común para mantener ambas representaciones.
    combined_features = layers.Concatenate(name="combined_final_features")([final_h1_features, final_h2_features])

    # Capa de salida para la clasificación/regresión
    output_layer = layers.Dense(output_units, activation='softmax', name="output_prediction")
    main_output = output_layer(combined_features)

    # --- Construir el Modelo Keras ---
    model = keras.Model(inputs=entrada_modelo, outputs=main_output, name="ODFE_NN")

    # --- Añadir la Pérdida de Decorrelación como una pérdida adicional ---
    # TensorFlow/Keras permite añadir pérdidas al modelo que no están directamente
    # asociadas con la salida principal, pero afectan a las activaciones internas.
    model.add_loss(decorrelation_loss(final_h1_features, final_h2_features, lambda_coeff=lambda_redundancia))

    return model

# --- Parámetros del Experimento ---
INPUT_DIM = 50           # Dimensiones de las características de entrada
UNITS_H1 = 64            # Neuronas en Capa Oculta 1
UNITS_H2 = 64            # Neuronas en Capa Oculta 2 (se asume mismo número para penalización de coseno simple)
OUTPUT_UNITS = 5         # Número de clases de salida
LAMBDA_REDUNDANCIA = 0.1 # Coeficiente para la pérdida de decorrelación

# --- Construir y Resumir el Modelo ---
print("--- Construyendo la Red Neuronal con Extracción de Características Orto-Diversas (ODFE-NN) ---")
model_odfe = construir_odfe_nn(
    input_shape=(INPUT_DIM,),
    units_h1=UNITS_H1,
    units_h2=UNITS_H2,
    output_units=OUTPUT_UNITS,
    lambda_redundancia=LAMBDA_REDUNDANCIA
)

model_odfe.summary()

# --- Preparar Datos Sintéticos para el Experimento ---
print("\n--- Generando datos sintéticos para el experimento ---")
num_samples = 1000
X_synthetic = np.random.rand(num_samples, INPUT_DIM).astype(np.float32)
y_synthetic = np.random.randint(0, OUTPUT_UNITS, num_samples)

# --- Compilar el Modelo ---
print("\n--- Compilando el modelo experimental ---")
# La pérdida principal es para la tarea de clasificación
model_odfe.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy', # Para clasificación
                   metrics=['accuracy'])

# --- Entrenar el Modelo ---
print("\n--- Iniciando el entrenamiento del modelo ODFE-NN ---")
history = model_odfe.fit(X_synthetic, y_synthetic,
                         epochs=10, # Más epochs para que la regularización tenga efecto
                         batch_size=32,
                         validation_split=0.2,
                         verbose=1)

print("\n--- Entrenamiento finalizado ---")

# --- Evaluar el Rendimiento ---
print("\n--- Evaluando el modelo ---")
X_test = np.random.rand(100, INPUT_DIM).astype(np.float32)
y_test = np.random.randint(0, OUTPUT_UNITS, 100)

loss, accuracy = model_odfe.evaluate(X_test, y_test, verbose=0)
print(f"Pérdida en datos de prueba: {loss:.4f}")
print(f"Precisión en datos de prueba: {accuracy:.4f}")

# --- Acceder a las activaciones internas (para ver la decorrelación) ---
print("\n--- Observando activaciones internas (requiere modelo auxiliar) ---")
# Crear un modelo auxiliar para obtener las activaciones de las capas intermedias
# Necesitamos acceder a las salidas internas de las capas ExtractorCapaUno y ExtractorCapaDos
# Esto se hace creando un nuevo modelo con esas salidas
extractor_model = keras.Model(inputs=model_odfe.input,
                              outputs=[model_odfe.get_layer("extractor_h1").output,
                                       model_odfe.get_layer("extractor_h2").output])

# Obtener activaciones para algunas muestras de prueba
h1_activations, h2_activations = extractor_model.predict(X_test[:5])

print(f"Activaciones H1 para las primeras 5 muestras:\n{h1_activations}")
print(f"Activaciones H2 para las primeras 5 muestras:\n{h2_activations}")

# Calcular y mostrar la similitud de coseno promedio entre las activaciones H1 y H2
# para ver si la penalización de redundancia está funcionando.
h1_norm_sample = tf.linalg.normalize(h1_activations, axis=-1)[0]
h2_norm_sample = tf.linalg.normalize(h2_activations, axis=-1)[0]
sample_similitud_coseno = tf.reduce_mean(tf.reduce_sum(h1_norm_sample * h2_norm_sample, axis=-1))
print(f"\nSimilitud de Coseno promedio entre H1 y H2 (muestras de prueba): {sample_similitud_coseno:.4f}")
# Un valor cercano a 0 indica baja similitud (ortogonalidad), lo que es el objetivo.

## Fusión contextual recíproca recurrente (RCF-NN)

Celda recurrente: en cada paso de la secuencia, H1 y H2 se pasan información. En el banco: arquitectura `rcf_recurrente` para series, contra la LSTM.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# --- 1. Definición de las Capas Personalizadas (adaptadas para RNN) ---

# Para una interacción bidireccional más directa, necesitaríamos diseñar
# una 'RNNCell' personalizada que maneje ambos flujos en cada paso.
# Esto es significativamente más complejo que el código actual.
# Aquí, voy a adaptar la idea a una estructura que Keras puede manejar,
# simulando la 'fusión' en cada paso de la secuencia.

class CustomRecurrentCell(layers.Layer):
    """
    Una celda recurrente personalizada que simula la interacción de H1 y H2.
    Esto es una simplificación extrema, un modelo real sería mucho más complejo.
    """
    def __init__(self, units_h1, units_h2, **kwargs):
        super(CustomRecurrentCell, self).__init__(**kwargs)
        self.units_h1 = units_h1
        self.units_h2 = units_h2

        # Capas para H1 (recibe input, estado H1 anterior, estado H2 anterior)
        self.dense_h1_input = layers.Dense(units_h1, activation='relu')
        self.dense_h1_prev_h1 = layers.Dense(units_h1, activation='relu')
        self.dense_h1_prev_h2 = layers.Dense(units_h1, activation='relu') # Para retroalimentación H2 -> H1

        # Capas para H2 (recibe input, estado H1 actual, estado H2 anterior)
        self.dense_h2_input = layers.Dense(units_h2, activation='relu')
        self.dense_h2_from_h1 = layers.Dense(units_h2, activation='relu')
        self.dense_h2_prev_h2 = layers.Dense(units_h2, activation='relu')

        self.combiner_h1 = layers.Concatenate(axis=-1)
        self.combiner_h2 = layers.Concatenate(axis=-1)

        # Una capa final para procesar la combinación en cada paso
        self.final_h1_transform = layers.Dense(units_h1, activation='relu')
        self.final_h2_transform = layers.Dense(units_h2, activation='relu')

        # Agregar el atributo state_size
        self.state_size = [self.units_h1, self.units_h2]


    def call(self, inputs, states):
        prev_h1_state, prev_h2_state = states

        # --- Cálculo para H1 ---
        # H1 recibe la entrada actual, el estado anterior de H1, Y el estado anterior de H2
        processed_input_h1 = self.dense_h1_input(inputs)
        processed_prev_h1 = self.dense_h1_prev_h1(prev_h1_state)
        processed_prev_h2_for_h1 = self.dense_h1_prev_h2(prev_h2_state) # Esto es la retroalimentación H2 -> H1

        combined_h1_inputs = self.combiner_h1([processed_input_h1, processed_prev_h1, processed_prev_h2_for_h1])
        current_h1_state = self.final_h1_transform(combined_h1_inputs)

        # --- Cálculo para H2 ---
        # H2 recibe la entrada actual, la salida ACTUAL de H1, y el estado anterior de H2
        # Para simular "salida ACTUAL de H1", usamos current_h1_state
        processed_input_h2 = self.dense_h2_input(inputs)
        processed_h1_for_h2 = self.dense_h2_from_h1(current_h1_state) # H1 -> H2
        processed_prev_h2 = self.dense_h2_prev_h2(prev_h2_state)

        combined_h2_inputs = self.combiner_h2([processed_input_h2, processed_h1_for_h2, processed_prev_h2])
        current_h2_state = self.final_h2_transform(combined_h2_inputs)

        # La salida de la celda es el estado de H2 (podría ser la concatenación de H1 y H2)
        # Los nuevos estados son los estados actuales de H1 y H2
        return current_h2_state, [current_h1_state, current_h2_state]

    def get_initial_state(self, inputs=None, batch_size=None, dtype=None):
        # Define los estados iniciales para H1 y H2
        # If dtype is None, use a default like tf.float32
        if dtype is None:
            dtype = tf.float32
        return [tf.zeros((batch_size, self.units_h1), dtype=dtype),
                tf.zeros((batch_size, self.units_h2), dtype=dtype)]

    def get_config(self):
        config = super().get_config()
        config.update({
            "units_h1": self.units_h1,
            "units_h2": self.units_h2,
        })
        return config

# --- 2. Construcción del Modelo con Capa RNN Personalizada ---

def construir_modelo_rcf_nn_recurrente(input_sequence_length, input_features, hidden_units_1, hidden_units_2, output_units):
    """
    Construye la Red Neuronal de Fusión Contextual Recíproca (RCF-NN)
    utilizando una capa RNN con una celda personalizada para procesar secuencias.
    """
    entrada_secuencia = keras.Input(shape=(input_sequence_length, input_features), name="secuencia_entrada")

    # Envuelve nuestra celda personalizada en una capa RNN
    # return_sequences=False significa que solo queremos la salida del último paso de tiempo
    # return_state=True significa que también obtendremos los estados finales de H1 y H2
    rnn_output, h1_final_state, h2_final_state = layers.RNN(
        CustomRecurrentCell(hidden_units_1, hidden_units_2),
        return_sequences=False, # Si necesitas una salida por cada paso, pon True
        return_state=True,      # Para obtener los estados finales de H1 y H2
        name="rcf_rnn_layer"
    )(entrada_secuencia)

    # La salida final puede ser el estado final de H2, o una combinación de ambos
    final_combined_state = layers.Concatenate(name="final_state_fusion")([h1_final_state, h2_final_state])

    salida = layers.Dense(output_units, activation='softmax', name="salida_prediccion")(final_combined_state)

    model = keras.Model(inputs=entrada_secuencia, outputs=salida, name="RCF_NN_Recurrente_Experimental")
    return model

# --- Parámetros del Experimento ---
SEQUENCE_LENGTH = 10    # Número de "momentos" o frames en cada secuencia
INPUT_FEATURES = 20     # Dimensiones de las características de cada momento
HIDDEN_1_UNITS = 32     # Neuronas en H1
HIDDEN_2_UNITS = 64     # Neuronas en H2
OUTPUT_UNITS = 5        # Número de clases de salida

# --- Construir y Resumir el Modelo Experimental ---
print("--- Construyendo la Red Neuronal de Fusión Contextual Recíproca (RCF-NN) Recurrente ---")
modelo_rcf_recurrente = construir_modelo_rcf_nn_recurrente(
    input_sequence_length=SEQUENCE_LENGTH,
    input_features=INPUT_FEATURES,
    hidden_units_1=HIDDEN_1_UNITS,
    hidden_units_2=HIDDEN_2_UNITS,
    output_units=OUTPUT_UNITS
)

modelo_rcf_recurrente.summary()

# --- Preparar Datos Sintéticos para el Experimento ---
print("\n--- Generando datos sintéticos para el experimento ---")
num_samples = 100 # Reducido para una ejecución más rápida en Colab
X_sintetico = np.random.rand(num_samples, SEQUENCE_LENGTH, INPUT_FEATURES).astype(np.float32)
y_sintetico = np.random.randint(0, OUTPUT_UNITS, num_samples)

# --- Compilar el Modelo para el Experimento ---
print("\n--- Compilando el modelo experimental ---")
modelo_rcf_recurrente.compile(optimizer='adam',
                              loss='sparse_categorical_crossentropy',
                              metrics=['accuracy'])

# --- Ejecutar el Experimento (Entrenamiento) ---
print("\n--- Iniciando el experimento de entrenamiento ---")
history = modelo_rcf_recurrente.fit(X_sintetico, y_sintetico,
                                    epochs=3, # Muy pocas epochs, solo para demostración
                                    batch_size=16,
                                    validation_split=0.2,
                                    verbose=1)

print("\n--- Experimento de entrenamiento finalizado ---")

## Reconstrucción de imágenes con tripletas

Una CNN de 3 capas ve el fotograma anterior y el siguiente y dibuja el del medio (interpolación de video). Se dejó una versión (la que separa validación). Medido en el banco (`laboratorio/banco/video.py`, resultados en `laboratorio/banco/resultados-04-video.md`).

In [ ]:
import torch
import os
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt # Para visualizar imágenes

# ---------------- CONFIGURACIÓN ---------------- #
FRAMES_PATH = "/content/npy_frames"
BATCH_SIZE = 1
MAX_EPOCHS = 50
SIMILARITY_THRESHOLD = 98.0
TARGET_BATCHES_PER_EPOCH = 10 # Cambiado a 100 para un ejemplo más realista

# Configuración para dividir el dataset
TRAIN_RATIO = 0.8 # 80% para entrenamiento
VAL_RATIO = 0.0001   # 10% para validación (se usa en evaluación)
TEST_RATIO = 0.1  # 10% para test (nueva sección de métricas de test)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
if use_amp:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
print(f"Usando dispositivo: {device}")
if use_amp:
    print("AMP (Automatic Mixed Precision) habilitado.")

try:
    primer_archivo = sorted([f for f in os.listdir(FRAMES_PATH) if f.endswith(".npy")])[0]
    primer_frame = np.load(os.path.join(FRAMES_PATH, primer_archivo))
    if primer_frame.ndim >= 3:
        CHANNELS = primer_frame.shape[2]
        # Almacenar alto y ancho para visualización
        HEIGHT, WIDTH = primer_frame.shape[0], primer_frame.shape[1]
    elif primer_frame.ndim == 2:
        CHANNELS = 1
        HEIGHT, WIDTH = primer_frame.shape[0], primer_frame.shape[1]
    else:
        raise ValueError("Formato de frame .npy no reconocido. Se espera (alto, ancho) o (alto, ancho, canales).")
except IndexError:
    raise FileNotFoundError(f"No se encontraron archivos .npy en la ruta: {FRAMES_PATH}. Asegúrate de que la carpeta no esté vacía.")
print(f"Detectados {CHANNELS} canal(es) con resolución {HEIGHT}x{WIDTH}.")


# ---------------- DATASET ---------------- #
class TripletasDataset(Dataset):
    def __init__(self, path):
        self.frames = sorted([os.path.join(path, f) for f in os.listdir(path) if f.endswith('.npy')])
        if len(self.frames) < 3:
            raise ValueError(f"Se necesitan al menos 3 frames para crear tripletas. Solo se encontraron {len(self.frames)}.")

    def __len__(self):
        return len(self.frames) - 2

    def __getitem__(self, idx):
        try:
            # Carga y preprocesamiento de los frames
            a = torch.tensor(np.load(self.frames[idx]), dtype=torch.float32).permute(2, 0, 1) / 255.0
            b = torch.tensor(np.load(self.frames[idx+1]), dtype=torch.float32).permute(2, 0, 1) / 255.0
            c = torch.tensor(np.load(self.frames[idx+2]), dtype=torch.float32).permute(2, 0, 1) / 255.0

            expected_channels = CHANNELS
            # Verifica el número de canales de los tensores
            if a.shape[0] != expected_channels or b.shape[0] != expected_channels or c.shape[0] != expected_channels:
                 raise ValueError(f"Discrepancia en el número de canales. Esperado: {expected_channels}, Obtenido: a={a.shape[0]}, b={b.shape[0]}, c={c.shape[0]}")

            # Verifica las dimensiones de la imagen (alto, ancho)
            if a.shape[1] != HEIGHT or a.shape[2] != WIDTH:
                raise ValueError(f"Discrepancia en la resolución de la imagen. Esperado: {HEIGHT}x{WIDTH}, Obtenido: {a.shape[1]}x{a.shape[2]}")

            return a, b, c
        except Exception as e:
            print(f"Error al cargar o procesar frame en índice {idx}: {e}")
            raise



# ---------------- MODELO ---------------- #
class ImageReconstructionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(CHANNELS * 2, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, CHANNELS, kernel_size=1)
        )

    def forward(self, x1, x2):
        return self.net(torch.cat([x1, x2], dim=1))



# ---------------- MÉTRICAS ---------------- #
def similarity_percentage(predicted, target):
    predicted_int = (predicted * 255).clamp(0, 255).byte()
    target_int = (target * 255).clamp(0, 255).byte()
    return (predicted_int == target_int).float().mean().item() * 100

def psnr(predicted, target):
    """
    Calcula el Peak Signal-to-Noise Ratio (PSNR).
    predicted y target deben estar en el rango [0, 1].
    """
    mse = torch.mean((predicted - target) ** 2).item()
    if mse == 0:
        return 100.0 # PSNR infinito para imágenes idénticas
    max_pixel = 1.0 # Para imágenes normalizadas a [0, 1]
    psnr_val = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr_val


# ---------------- VISUALIZACIÓN ---------------- #
def visualize_image(image_tensor, title="Imagen Generada"):
    """
    Visualiza un tensor de imagen.
    image_tensor: Tensor de PyTorch (C, H, W) normalizado a [0, 1].
    """
    # Mueve el tensor a la CPU y lo convierte a numpy
    image_np = image_tensor.detach().cpu().numpy()

    # Permuta las dimensiones a (H, W, C) para matplotlib
    if CHANNELS == 1: # Si es escala de grises, quita la dimensión de canal
        image_np = image_np.squeeze()
        plt.imshow(image_np, cmap='gray')
    else: # RGB o multi-canal
        image_np = np.transpose(image_np, (1, 2, 0))
        plt.imshow(image_np)

    plt.title(title)
    plt.axis('off')
    plt.show()



# ---------------- ENTRENAMIENTO ---------------- #
def train_model(model, train_loader, val_loader, optimizer, criterion):
    # Calcular el número total de batches disponibles en el set de entrenamiento
    total_train_batches_available = len(train_loader)
    batches_to_process = TARGET_BATCHES_PER_EPOCH if TARGET_BATCHES_PER_EPOCH is not None else total_train_batches_available

    print(f"Se procesarán {batches_to_process} de {total_train_batches_available} batches de entrenamiento por época.")

    last_generated_image = None # Para almacenar la última imagen generada para visualización

    for epoch in range(MAX_EPOCHS):
        model.train()
        pbar = tqdm(enumerate(train_loader), total=batches_to_process, desc=f"Época {epoch+1:02d} [Entrenamiento]")

        last_loss = 0.0

        for batch_idx, (a, b, c) in pbar:
            if batch_idx >= batches_to_process:
                print(f"Límite de {batches_to_process} batches alcanzado en la época {epoch+1}.")
                break

            a, b, c = a.to(device), b.to(device), c.to(device)
            optimizer.zero_grad()

            if use_amp:
                with autocast():
                    y = model(a, c)
                    loss = criterion(y, b)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                y = model(a, c)
                loss = criterion(y, b)
                loss.backward()
                optimizer.step()

            last_loss = loss.item()
            pbar.set_postfix(loss=last_loss)

            # Guardar la última imagen generada del batch para visualización
            # Tomamos la primera imagen del batch
            last_generated_image = y[0]

        # Evaluación en el conjunto de validación
        model.eval()
        with torch.no_grad():
            eval_batches_count = 0
            total_sim_val = 0.0
            total_psnr_val = 0.0 # Nueva métrica PSNR para validación

            eval_pbar = tqdm(val_loader, desc="Evaluando en validación")
            for u, v, w in eval_pbar:
                u, v, w = u.to(device), v.to(device), w.to(device)
                z = model(u, w)
                total_sim_val += similarity_percentage(z, v)
                total_psnr_val += psnr(z, v) # Suma PSNR
                eval_batches_count += 1

            if eval_batches_count > 0:
                avg_similarity_val = total_sim_val / eval_batches_count
                avg_psnr_val = total_psnr_val / eval_batches_count # Promedio PSNR
                print(f"Época {epoch+1:02d} - Similitud promedio (Validación): {avg_similarity_val:.2f}% - PSNR (Validación): {avg_psnr_val:.2f}dB - Última Pérdida (Entrenamiento): {last_loss:.4f}")
                if avg_similarity_val >= SIMILARITY_THRESHOLD:
                    print("Umbral de similitud alcanzado. Deteniendo el entrenamiento.")
                    break
            else:
                print("No se pudieron realizar batches para evaluación en esta época.")

        # Visualizar la última imagen generada al final de cada época
        if last_generated_image is not None:
            # Se muestra la imagen generada (y) y el objetivo (b) de la última iteración
            # Para mayor claridad, puedes obtener un batch de ejemplo al final de la época para visualizar
            # Aquí, reusaremos u, v, w del último batch de validación
            if 'u' in locals() and 'v' in locals() and 'w' in locals():
                model.eval()
                with torch.no_grad():
                    sample_predicted = model(u[0:1].to(device), w[0:1].to(device)).squeeze(0) # Obtener una sola predicción
                    sample_target = v[0].squeeze(0) # Obtener el target correspondiente

                print("\nVisualizando la última imagen generada:")
                visualize_image(sample_predicted, title=f"Predicción (Época {epoch+1})")
                visualize_image(sample_target, title=f"Target Real (Época {epoch+1})")

                # Opcional: Visualizar la imagen de entrada (a y c)
                # visualize_image(u[0].squeeze(0), title=f"Input 'a' (Época {epoch+1})")
                # visualize_image(w[0].squeeze(0), title=f"Input 'c' (Época {epoch+1})")



# ---------------- FUNCIÓN DE TEST ---------------- #
def test_model(model, test_loader, criterion):
    model.eval() # Poner el modelo en modo de evaluación
    total_test_loss = 0.0
    total_test_similarity = 0.0
    total_test_psnr = 0.0
    test_batches_count = 0

    print("\n--- Iniciando Evaluación en el Conjunto de Test ---")
    test_pbar = tqdm(test_loader, desc="Evaluando en Test")

    with torch.no_grad(): # Deshabilitar cálculo de gradientes
        for a, b, c in test_pbar:
            a, b, c = a.to(device), b.to(device), c.to(device)

            y = model(a, c) # Predicción
            loss = criterion(y, b) # Pérdida

            total_test_loss += loss.item()
            total_test_similarity += similarity_percentage(y, b)
            total_test_psnr += psnr(y, b)
            test_batches_count += 1

    if test_batches_count > 0:
        avg_test_loss = total_test_loss / test_batches_count
        avg_test_similarity = total_test_similarity / test_batches_count
        avg_test_psnr = total_test_psnr / test_batches_count
        print(f"\n--- Resultados Finales en Test ---")
        print(f"Pérdida promedio (Test): {avg_test_loss:.4f}")
        print(f"Similitud promedio (Test): {avg_test_similarity:.2f}%")
        print(f"PSNR promedio (Test): {avg_test_psnr:.2f}dB")
    else:
        print("No se encontraron datos en el conjunto de test para evaluar.")



# ---------------- MAIN ---------------- #
if __name__ == "__main__":
    # 1. Cargar el dataset completo
    full_dataset = TripletasDataset(FRAMES_PATH)

    # 2. Dividir el dataset en entrenamiento, validación y test
    # Asegúrate de que las sumas de los ratios sean 1.0 (o muy cercanas por errores de coma flotante)
    train_size = int(TRAIN_RATIO * len(full_dataset))
    val_size = int(VAL_RATIO * len(full_dataset))
    test_size = len(full_dataset) - train_size - val_size # Asegura que la suma sea el total

    if train_size == 0 or val_size == 0 or test_size == 0:
        raise ValueError("Uno de los conjuntos de datos (entrenamiento, validación o test) es demasiado pequeño o vacío. Ajusta los ratios o el número total de frames.")

    train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

    print(f"Dataset dividido: Entrenamiento={len(train_dataset)} | Validación={len(val_dataset)} | Test={len(test_dataset)}")

    # 3. Crear DataLoaders para cada conjunto
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=os.cpu_count() // 2 or 1)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=os.cpu_count() // 2 or 1) # No shuffule en validación
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=os.cpu_count() // 2 or 1) # No shuffule en test

    # 4. Instanciar el modelo, optimizador y criterio de pérdida
    model = ImageReconstructionModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    # 5. Iniciar el entrenamiento
    train_model(model, train_loader, val_loader, optimizer, criterion)

    # 6. Ejecutar la evaluación final en el conjunto de test
    test_model(model, test_loader, criterion)

    # Opcional: Guardar el modelo entrenado
    # torch.save(model.state_dict(), "modelo_entrenado.pth")
    # print("Modelo guardado como 'modelo_entrenado.pth'.")

## FrameInterpolator: el fotograma del medio a partir de 4

Ve los fotogramas i, i+1, i+3 e i+4 y dibuja el i+2 con una LSTM sobre la imagen aplanada. Tu cuaderno completo de Drive tenía la misma red con precisión mixta (más rápida en GPU). Medido en el banco (`laboratorio/banco/video.py`, resultados en `laboratorio/banco/resultados-04-video.md`).

In [ ]:
# Correr en Google Colab
import torch
import torch.nn as nn
import torch.optim as optim
import imageio
import cv2
import numpy as np
import os # Importar os para os.cpu_count()
from tqdm import tqdm # Importar tqdm para barras de progreso visual
import matplotlib.pyplot as plt # Importar matplotlib para visualización
import random # Importar random para get_batch en el ejemplo de Transformer (aunque no se usa en el modelo actual)


# ---------------- CONFIGURACIÓN ---------------- #
# Usar variables de entorno o un archivo de configuración (ej. .ini, YAML)
# para una mejor gestión y flexibilidad en diferentes entornos.
video_path = '/content/mi_video.mp4'  # Ruta a tu video
width, height = 160, 90   # Resolución reducida
max_episodes = 50
max_frames = 3000 # Número máximo de frames a cargar del video
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# use_amp: Automatic Mixed Precision. Habilita si usas GPU para un entrenamiento más rápido.
# Requiere que el dispositivo sea 'cuda'.
use_amp = device.type == 'cuda'
if use_amp:
    # GradScaler es necesario para AMP para escalar gradientes y evitar underflow.
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
print(f"Usando dispositivo: {device}")
if use_amp:
    print("AMP (Automatic Mixed Precision) habilitado.")


# ---------------- CARGA DE DATOS Y PREPROCESAMIENTO ---------------- #
def cargar_video(path, max_frames=30):
    """
    Carga frames de un video, los redimensiona y convierte a tensor PyTorch.
    Asegura la carga en formato RGB (3 canales).
    """
    frames = []
    try:
        # imageio.get_reader maneja varios formatos de video.
        # Usar 'ffmpeg' como plugin explícitamente puede ayudar si hay problemas.
        reader = imageio.get_reader(path, 'ffmpeg')
        print(f"Cargando video: {path}")
        # Obtener metadatos del video si es posible
        # meta = reader.get_meta_data()
        # print(f"Metadata: {meta}")

        for i, frame in enumerate(reader):
            if i >= max_frames:
                print(f"Límite de {max_frames} frames alcanzado.")
                break
            # frame viene como numpy array, típicamente en formato HxWxChannels (uint8)
            # Asegurarse de que sea RGB si es necesario (algunos readers pueden dar BGR)
            # OpenCV resize devuelve BGR por defecto para imágenes en color.
            # Aunque imageio reader ya puede dar RGB, es una buena práctica ser explícito si se usa cv2.resize.
            # Sin embargo, para este caso, imageio's reader suele dar RGB con el plugin ffmpeg.
            # Si el frame no tiene 3 dimensiones, significa que no es color (ej. escala de grises, transparencia).
            # Descartamos frames que no sean RGB para asegurar consistencia.
            if frame.ndim == 3 and frame.shape[2] == 3:
                # Redimensionar el frame al tamaño deseado (width, height)
                # cv2.resize espera (ancho, alto) en la tupla de tamaño.
                resized_frame = cv2.resize(frame, (width, height))

                # Convertir de numpy a tensor PyTorch.
                # Convertir uint8 a float32 y normalizar a [0, 1].
                tensor = torch.tensor(resized_frame, dtype=torch.float32) / 255.0

                # Permutar dimensiones de (H, W, C) a (C, H, W) para PyTorch Conv2d
                tensor = tensor.permute(2, 0, 1) # Permute to [3, H, W]

                frames.append(tensor)
            else:
                print(f"Saltando frame {i} debido a formato no compatible (esperado RGB 3 canales, got {frame.shape}).")


        reader.close() # Cerrar el reader para liberar recursos

    except imageio.core.fetching.NeedDownloadError as e:
         print(f"Error: Necesitas descargar dependencias (ej. ffmpeg). Intenta ejecutar !pip install imageio[ffmpeg]. Error: {e}")
         return None # Retornar None para indicar fallo
    except imageio.core.format.CannotOpen as e:
        print(f"Error: No se pudo abrir o leer el video en '{path}'. Asegúrate de que la ruta es correcta y el archivo no está corrupto. Error: {e}")
        return None
    except Exception as e:
        print(f"Ocurrió un error inesperado al cargar el video: {e}")
        return None

    if not frames:
        print(f"No se pudieron cargar frames del video en '{path}'. La lista de frames está vacía.")
        return None

    # Apilar todos los tensores de frame en un único tensor
    # El resultado tendrá forma [Número_de_frames, Canales, Alto, Ancho]
    return torch.stack(frames)


# ---------------- MÉTRICAS ---------------- #
def similitud_color(predicted, target):
    """
    Calcula la similitud entre dos tensores de imagen usando PSNR.
    predicted y target deben estar en el rango [0, 1].
    Retorna el valor de PSNR en dB.
    """
    # Asegurar que los tensores están en el mismo dispositivo y tipo de dato float
    predicted = predicted.float()
    target = target.float()

    # Calcular el error cuadrático medio (MSE)
    # El promedio se calcula sobre todos los elementos (pixels) en el tensor.
    mse = torch.mean((predicted - target) ** 2)

    # Si el MSE es cero (imágenes idénticas), PSNR es infinito.
    if mse == 0:
        return 100.0 # Un valor alto para representar PSNR infinito

    # Calcular el PSNR
    max_pixel = 1.0 # Para imágenes normalizadas a [0, 1]
    # PSNR = 10 * log10(MAX^2 / MSE)
    # Evitar log de cero si MSE es muy pequeño pero no cero, usando un pequeño epsilon.
    psnr_val = 10 * torch.log10(max_pixel**2 / (mse + 1e-8)) # Agregar epsilon
    return psnr_val.item() # Retornar el valor numérico

# ---------------- VISUALIZACIÓN ---------------- #
def visualize_image(image_tensor, title="Imagen Generada"):
    """
    Visualiza un tensor de imagen.
    image_tensor: Tensor de PyTorch (C, H, W) normalizado a [0, 1].
    Asume C=3 (RGB) para mostrar en color. Si C=1, lo muestra en escala de grises.
    """
    if image_tensor is None:
        print(f"No se puede visualizar: El tensor para '{title}' es None.")
        return

    # Mover el tensor a la CPU y convertir a numpy array
    # .detach() crea una nueva copia del tensor que no requiere gradiente.
    # .cpu() mueve el tensor a la CPU si está en la GPU.
    image_np = image_tensor.detach().cpu().numpy()

    # Permutar las dimensiones de (C, H, W) a (H, W, C) para que matplotlib pueda mostrarlo.
    # Para imágenes en escala de grises (C=1), matplotlib imshow puede tomar (H, W) o (H, W, 1).
    if image_np.shape[0] == 1: # Escala de grises
        image_np = image_np.squeeze(0) # Remover la dimensión de canal
        plt.imshow(image_np, cmap='gray')
    elif image_np.shape[0] == 3: # RGB
        image_np = np.transpose(image_np, (1, 2, 0)) # Permutar a (H, W, C)
        # Asegurarse de que los valores están en el rango [0, 1] para imshow con float32
        image_np = np.clip(image_np, 0, 1)
        plt.imshow(image_np)
    else:
        print(f"No se puede visualizar: Formato de canal desconocido {image_np.shape[0]} para '{title}'. Esperado 1 o 3.")
        return

    plt.title(title)
    plt.axis('off') # Ocultar ejes
    plt.show() # Mostrar la imagen

# ---------------- MODELO ---------------- #
class FrameInterpolator(nn.Module):
    def __init__(self, height, width):
        super().__init__()
        self.height = height
        self.width = width
        # El modelo espera como entrada 4 frames concatenados a lo largo del canal.
        # Cada frame es (3, H, W). Concatenar 4 de ellos da (12, H, W).
        # Este tensor de (12, H, W) se aplana a (12 * H * W) características.
        # La capa LSTM procesará este vector de características como una única "secuencia" de longitud 1.
        lstm_input_size = self.height * self.width * 3 * 4 # 4 frames * 3 canales * H * W
        lstm_hidden_size = 1024 # Tamaño de la dimensión oculta del LSTM
        num_lstm_layers = 2 # Número de capas del LSTM

        # Definir la capa LSTM
        # batch_first=True significa que la forma de entrada es (Batch, Sequence Length, Features)
        self.lstm = nn.LSTM(input_size=lstm_input_size,
                            hidden_size=lstm_hidden_size,
                            num_layers=num_lstm_layers,
                            batch_first=True)

        # Capa Lineal que toma la salida de la última capa oculta del LSTM
        # y la mapea al tamaño de un frame plano (3 * H * W).
        self.fc = nn.Linear(lstm_hidden_size, self.height * self.width * 3)

    def forward(self, x):
        # x es el tensor de entrada que contiene 4 frames concatenados.
        # Su forma es (Batch, 12, Height, Width).
        batch_size = x.size(0)

        # Aplanar el tensor de entrada (excepto la dimensión del batch).
        # Esto combina los 12 canales y las dimensiones espaciales HxW en un único vector de características.
        # La forma resultante es (Batch, 12 * Height * Width).
        x = x.view(batch_size, -1) # Aplana a (Batch, 12 * H * W)

        # El LSTM espera una dimensión de longitud de secuencia.
        # Como estamos tratando los 4 frames combinados como una sola "observación" o paso de tiempo,
        # la longitud de la secuencia es 1. Añadimos esta dimensión.
        # La forma resultante es (Batch, 1, 12 * Height * Width).
        x = x.unsqueeze(1) # Añade la dimensión de secuencia: (Batch, 1, Features)

        # Pasar el tensor a través de la capa LSTM.
        # El LSTM retorna (output, (hidden_state, cell_state)).
        # `output` tiene forma (Batch, Seq Len, Hidden Size), pero como Seq Len es 1, es (Batch, 1, Hidden Size).
        # `hidden_state` tiene forma (Num Layers, Batch, Hidden Size).
        # `cell_state` tiene forma (Num Layers, Batch, Hidden Size).
        # Solo nos interesa el estado oculto final `h` (de todas las capas).
        _, (h, _) = self.lstm(x)

        # Tomar el estado oculto de la última capa del LSTM.
        # h[-1] selecciona el estado oculto de la última capa, con forma (Batch, Hidden Size).
        lstm_out = h[-1]

        # Pasar la salida del LSTM a través de la capa lineal.
        # La capa lineal mapea (Batch, Hidden Size) a (Batch, 3 * Height * Width).
        out = self.fc(lstm_out)

        # Remodelar la salida plana (Batch, 3 * Height * Width) de nuevo a la forma de imagen.
        # La forma deseada es (Batch, 3, Height, Width) para que coincida con el target.
        # -1 en .view() infiere el tamaño restante (3 * H * W).
        # Los otros argumentos (3, self.height, self.width) definen las dimensiones esperadas.
        return out.view(batch_size, 3, self.height, self.width)


# ---------------- FUNCIÓN DE ENTRENAMIENTO ---------------- #
def train(model, video, optimizer, loss_fn, device, max_episodes=50):
    """
    Función para entrenar el modelo de interpolación de frames.
    Procesa el video frame a frame, creando tripletas (a, b, c) donde 'b' es el target.
    Para este modelo LSTM, el input 'a' se expande para incluir frames i, i+1, i+3, i+4.
    """
    # Asegurarse de que el modelo y los datos estén en el dispositivo correcto
    model.to(device)
    video = video.to(device)

    # total_frames es el número de frames cargados del video
    total_frames = video.size(0)
    # Se necesita al menos 5 frames para formar la primera secuencia de entrada (i, i+1, i+3, i+4)
    # y el target (i+2). La última posible secuencia de entrada termina en frame total_frames - 1,
    # lo que significa que el primer frame de la secuencia es total_frames - 5.
    # El bucle va desde el índice 0 hasta total_frames - 5.
    num_sequences = total_frames - 4
    if num_sequences <= 0:
         print("No hay suficientes frames para crear secuencias de entrenamiento (se necesitan al menos 5).")
         return

    print(f"Total de secuencias de entrenamiento (i, i+1, i+3, i+4 -> i+2): {num_sequences}")


    for episode in range(max_episodes):
        # En un contexto de RL, "episode" a menudo significa un ciclo completo.
        # Aquí, lo usamos como sinónimo de época de entrenamiento sobre los datos disponibles.
        print(f"--- Episodio {episode+1}/{max_episodes} ---")

        # Barajar los índices de inicio de las secuencias para entrenar en un orden aleatorio
        # Esto simula un shuffle del dataset.
        indices = list(range(num_sequences))
        random.shuffle(indices)

        total_episode_loss = 0.0
        total_episode_sim = 0.0
        total_episode_psnr = 0.0
        processed_sequences = 0

        # Usar tqdm para mostrar una barra de progreso para las secuencias en el episodio
        pbar = tqdm(indices, desc=f"Entrenando Episodio {episode+1}")

        for i in pbar:
            # Extraer los frames para la secuencia de entrada y el target
            # input_frames: [video[i], video[i + 1], video[i + 3], video[i + 4]]
            # target: video[i + 2]

            # Asegurarse de que los índices están dentro de los límites del video cargado
            if i + 4 >= total_frames:
                 # Esto no debería ocurrir si el rango del bucle es correcto (hasta num_sequences - 1)
                 # pero es una buena práctica para evitar errores inesperados.
                 print(f"Saltando índice {i}: Fuera de los límites.")
                 continue

            # Preparar la entrada para el modelo
            # Concatenamos los 4 frames de entrada a lo largo de la dimensión de canal
            # Cada frame es [3, H, W]. Concatenar 4 da [12, H, W].
            input_frames_sequence = torch.cat([video[i], video[i + 1], video[i + 3], video[i + 4]], dim=0)

            # El modelo espera un batch, incluso si es de tamaño 1. Añadimos una dimensión de batch.
            # Forma final de la entrada para el modelo: [1, 12, H, W]
            input_for_model = input_frames_sequence.unsqueeze(0).to(device)

            # Preparar el target
            # El target es el frame i+2. Forma [3, H, W].
            # Añadimos una dimensión de batch para que coincida con la salida del modelo.
            # Forma final del target: [1, 3, H, W]
            target_frame = video[i + 2].unsqueeze(0).to(device)


            # ---- Proceso de Entrenamiento ----
            optimizer.zero_grad() # Poner a cero los gradientes

            # Usar autocast para AMP si está habilitado
            if use_amp:
                with autocast(): # Ejecuta el código en precision mixta
                    output = model(input_for_model) # Forward pass
                    loss = loss_fn(output, target_frame) # Calcular la pérdida

                scaler.scale(loss).backward() # Escalar la pérdida y calcular gradientes
                scaler.step(optimizer) # Actualizar pesos
                scaler.update() # Actualizar el escalador para la siguiente iteración
            else:
                output = model(input_for_model) # Forward pass
                loss = loss_fn(output, target_frame) # Calcular la pérdida

                loss.backward() # Backward pass (calcular gradientes)
                optimizer.step() # Actualizar pesos del modelo

            # ---- Calcular Métricas ----
            # Desnormalizar la salida y el target si es necesario para métricas (PSNR espera [0,1])
            # Nuestras métricas (similitud_color) ya esperan [0,1].
            sim = similitud_color(output.squeeze(0), target_frame.squeeze(0)) # Remover batch dim para la métrica

            total_episode_loss += loss.item() # Sumar la pérdida del batch
            total_episode_sim += sim # Sumar la similitud del batch
            processed_sequences += 1

            # Actualizar la barra de progreso con la pérdida actual
            pbar.set_postfix(loss=loss.item(), psnr=sim)


        # ---- Resumen del Episodio ----
        avg_loss = total_episode_loss / processed_sequences if processed_sequences > 0 else 0
        avg_sim = total_episode_sim / processed_sequences if processed_sequences > 0 else 0

        print(f"Episodio {episode+1}/{max_episodes} Completado.")
        print(f"  Pérdida Promedio: {avg_loss:.4f}")
        print(f"  Similitud Promedio (PSNR): {avg_sim:.2f} dB")

        # Opcional: Visualizar un ejemplo al final de cada episodio
        if processed_sequences > 0:
             print("\nVisualizando un ejemplo del último batch procesado:")
             # Usamos los tensores del último batch procesado dentro del bucle for i in pbar
             # Asegurarse de que estas variables (input_for_model, target_frame, output)
             # retienen el valor del último bucle.
             visualize_image(input_for_model.squeeze(0)[0:3], title=f"Input Frame i (Episodio {episode+1})") # Mostrar el primer frame de entrada (i)
             visualize_image(input_for_model.squeeze(0)[3:6], title=f"Input Frame i+1 (Episodio {episode+1})") # Mostrar el segundo frame de entrada (i+1)
             visualize_image(input_for_model.squeeze(0)[6:9], title=f"Input Frame i+3 (Episodio {episode+1})") # Mostrar el tercer frame de entrada (i+3)
             visualize_image(input_for_model.squeeze(0)[9:12], title=f"Input Frame i+4 (Episodio {episode+1})") # Mostrar el cuarto frame de entrada (i+4)
             visualize_image(output.squeeze(0), title=f"Predicción (Episodio {episode+1})")
             visualize_image(target_frame.squeeze(0), title=f"Target Real (Episodio {episode+1})")


# ---------------- MAIN ---------------- #
if __name__ == "__main__":
    # Asegurarse de que OpenCV y imageio están instalados
    # !pip install opencv-python-headless imageio imageio-ffmpeg tqdm matplotlib torch numpy

    # 1. Cargar y preprocesar el video
    # La función cargar_video retorna None si falla.
    video_tensor = cargar_video(video_path, max_frames)

    if video_tensor is None or video_tensor.size(0) < 5:
        print("Error: No se pudo cargar el video o no hay suficientes frames (se necesitan al menos 5). Saliendo.")
    else:
        print(f"Video cargado con forma: {video_tensor.shape} (Frames, Canales, Alto, Ancho)")
        num_loaded_frames = video_tensor.size(0)
        print(f"Número de frames cargados: {num_loaded_frames}")

        # 2. Instanciar el modelo, optimizador y criterio de pérdida
        # Pasar las dimensiones al modelo para que calcule correctamente el input_size
        model = FrameInterpolator(height, width).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        loss_fn = nn.MSELoss() # Mean Squared Error como función de pérdida

        # Imprimir un resumen del modelo
        # print(model)
        # print(f"Número de parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")


        # 3. Iniciar el entrenamiento
        # Pasar el tensor de video, no un DataLoader, ya que el entrenamiento
        # aquí itera sobre los frames directamente para construir las secuencias.
        train(model, video_tensor, optimizer, loss_fn, device, max_episodes)

        # Opcional: Guardar el modelo entrenado
        # torch.save(model.state_dict(), "modelo_interpolador_frames.pth")
        # print("Modelo guardado como 'modelo_interpolador_frames.pth'.")

        # Opcional: Cargar un modelo entrenado más tarde
        # model = FrameInterpolator(height, width).to(device)
        # model.load_state_dict(torch.load("modelo_interpolador_frames.pth", weights_only=True))
        # model.eval() # Poner el modelo en modo evaluación si no vas a entrenar más

        # --- Ejemplo de uso después del entrenamiento (Predicción de un frame) ---
        # Esto asume que tienes suficientes frames cargados para formar una secuencia de entrada.
        if num_loaded_frames >= 5:
            print("\n--- Ejemplo de Predicción después del Entrenamiento ---")
            model.eval() # Poner el modelo en modo evaluación (deshabilita dropout, batchnorm, etc.)
            with torch.no_grad(): # Deshabilitar cálculo de gradientes para la inferencia
                 # Seleccionar una secuencia de ejemplo (ej. la primera disponible)
                 i = 0
                 # Asegurarse de que i+4 no excede el número de frames cargados
                 if i + 4 < num_loaded_frames:
                     frame_i = video_tensor[i]
                     frame_i_plus_1 = video_tensor[i + 1]
                     frame_i_plus_3 = video_tensor[i + 3]
                     frame_i_plus_4 = video_tensor[i + 4]
                     target_frame = video_tensor[i + 2] # El frame que queremos predecir

                     # Preparar la entrada como en el entrenamiento
                     input_frames_sequence = torch.cat([frame_i, frame_i_plus_1, frame_i_plus_3, frame_i_plus_4], dim=0)
                     input_for_model = input_frames_sequence.unsqueeze(0).to(device)

                     # Realizar la predicción
                     predicted_frame = model(input_for_model) # [1, 3, H, W]

                     # Mover la predicción y el target a la CPU para visualización y métricas
                     predicted_frame = predicted_frame.squeeze(0).cpu() # Remover batch dim
                     target_frame = target_frame.cpu() # Ya está en CPU si video_tensor no se movió, pero mejor ser explícito

                     # Calcular métricas de la predicción
                     sim_pred = similitud_color(predicted_frame, target_frame)
                     print(f"PSNR de la predicción de ejemplo vs target real: {sim_pred:.2f} dB")

                     # Visualizar los frames de entrada, la predicción y el target
                     visualize_image(frame_i.cpu(), title="Input Frame i (Ejemplo Pred.)")
                     visualize_image(frame_i_plus_1.cpu(), title="Input Frame i+1 (Ejemplo Pred.)")
                     visualize_image(frame_i_plus_3.cpu(), title="Input Frame i+3 (Ejemplo Pred.)")
                     visualize_image(frame_i_plus_4.cpu(), title="Input Frame i+4 (Ejemplo Pred.)")
                     visualize_image(predicted_frame, title="Predicción (Ejemplo Pred.)")
                     visualize_image(target_frame, title="Target Real (Ejemplo Pred.)")
                 else:
                     print(f"No hay suficientes frames disponibles para la predicción de ejemplo empezando en índice {i}.")
        else:
            print("No hay suficientes frames en el video cargado para realizar una predicción de ejemplo.")

## CNN con imágenes marcadas

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image

# Función para procesar la imagen con las marcas del usuario
def procesar_imagen_marcada(image_editor_output):
    if image_editor_output is None:
        return "No se ha subido ninguna imagen o no se han hecho marcas."

    # La salida de ImageEditor es un diccionario.
    # 'composite' es la imagen final con todas las capas (fondo + marcas)
    # 'layers' es una lista de las capas individuales de dibujo
    composite_image = image_editor_output['composite']
    layers = image_editor_output['layers']

    if composite_image is None:
        return "No se pudo obtener la imagen compuesta."

    # Convertir la imagen compuesta a NumPy para procesamiento
    img_np = np.array(composite_image)

    # Identificar los puntos marcados
    # Si el usuario usa el pincel, la capa será la "pintura"
    # Vamos a buscar píxeles no blancos en las capas de dibujo.
    puntos_encontrados = []
    if layers:
        for layer_info in layers:
            layer_img = layer_info['image'] # Esto es una PIL Image
            layer_np = np.array(layer_img.convert('RGBA')) # Asegurarse de tener canal alfa

            # Encontrar píxeles que no son completamente transparentes (canal alfa > 0)
            # y que no son blancos (si el pincel es de color)
            # Para un pincel negro, buscaremos píxeles con valores bajos
            # Simplificación: si el pincel es negro, buscaremos píxeles negros
            # Si el pincel es de cualquier color, buscaremos píxeles que no sean la imagen original

            # Una forma más robusta sería comparar con la imagen original si no hay marcas.
            # Para este ejemplo, asumiremos que las marcas son de un color distintivo

            # Gradio ImageEditor las marcas suelen ser negras o de color.
            # Vamos a buscar píxeles que no sean (255, 255, 255, 0) o (0,0,0,0) (totalmente transparente)

            # Sumar los canales RGB para saber si es blanco o negro (más robusto para colores)
            # Los píxeles marcados tendrán un valor de color diferente al fondo (si es blanco)
            # y un canal alfa que indica opacidad.

            # Identificar píxeles que son significativamente diferentes del blanco puro transparente
            # y que tienen cierta opacidad.
            # Un umbral simple para identificar "marcas" que no son el color de fondo.

            # Una forma sencilla es buscar los píxeles que no son blancos (255,255,255) y son opacos
            # O simplemente donde el canal alfa no es 0

            non_transparent_pixels = np.where(layer_np[:, :, 3] > 0)

            for y, x in zip(non_transparent_pixels[0], non_transparent_pixels[1]):
                # Aquí podrías refinar la detección si solo quieres "puntos" y no "líneas"
                # Por ahora, cada píxel marcado es un punto.
                puntos_encontrados.append((x, y)) # (x, y) para coordenadas

    # Devolver los puntos encontrados o un mensaje
    if puntos_encontrados:
        # Eliminar duplicados si el usuario arrastró el pincel
        puntos_unicos = sorted(list(set(puntos_encontrados)))
        texto_puntos = "Puntos señalados:\n" + "\n".join([f"({x}, {y})" for x, y in puntos_unicos])
        return texto_puntos
    else:
        return "No se señalaron puntos."

# Interfaz Gradio
with gr.Blocks() as demo:
    gr.Markdown("# Señalar Puntos en Imagen con Gradio")
    gr.Markdown("Sube una imagen y usa la herramienta de pincel para señalar puntos.")

    with gr.Row():
        # gr.ImageEditor permite dibujar con el pincel
        # Los botones de herramientas permiten cambiar de herramienta
        # Eliminados brush_color y brush_size ya que no son argumentos del constructor en esta versión
        image_input = gr.ImageEditor(
            # tool="brush", # Herramienta inicial seleccionada: pincel (Eliminado)
            type="pil", # Para obtener la imagen como PIL Image en la función
            label="Sube una imagen y usa el pincel para marcar puntos"
        )

        output_text = gr.Textbox(label="Coordenadas de los puntos")

    # Cuando el ImageEditor cambia (al soltar el ratón después de pintar, por ejemplo)
    # se llama a la función procesar_imagen_marcada
    image_input.change(
        fn=procesar_imagen_marcada,
        inputs=image_input,
        outputs=output_text
    )

# Unindent this line to be outside the with block
demo.launch(debug=True) # debug=True te da más información en la consola de Colab
# Comentado para evitar que se lance automáticamente en este bloque.
# Puedes descomentar si quieres lanzar la interfaz desde aquí.

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
from scipy.ndimage import measurements # Necesitamos esto para calcular centroides

# Función para procesar la imagen con las marcas del usuario
def procesar_imagen_marcada(image_editor_output):
    if image_editor_output is None or image_editor_output['composite'] is None:
        return "No se ha subido ninguna imagen o no se han hecho marcas."

    # image_editor_output['layers'] contiene ahora directamente los objetos PIL Image
    # o None si no hay capas
    layers = image_editor_output['layers']

    puntos_señalados = []

    if layers:
        # Iterar directamente sobre las imágenes en 'layers'
        for i, layer_img_pil in enumerate(layers):
            if layer_img_pil is None:
                continue # Saltar si la capa es nula por alguna razón

            # Convertir la capa a un array NumPy en escala de grises para el procesamiento
            # Asumimos que las marcas son más oscuras que el fondo de la capa
            layer_np_gray = np.array(layer_img_pil.convert('L')) # 'L' para escala de grises

            # Queremos encontrar los píxeles que son parte de la marca (no el fondo blanco o transparente)
            # Un umbral simple: los píxeles oscuros son la marca
            # 255 es blanco, 0 es negro. Buscamos valores que no sean blancos (pincel oscuro).
            # Ajusta el umbral si tu pincel no es negro o si el fondo no es blanco
            marked_region = (layer_np_gray < 250).astype(int)

            # Si hay píxeles marcados en esta capa, calcula su centroide
            if np.sum(marked_region) > 0: # Solo si hay algo marcado en la capa
                # center_of_mass devuelve (y, x)
                # Manejar el caso en que center_of_mass pueda retornar NaN si marked_region es completamente cero (aunque el if anterior lo previene)
                try:
                    center_y, center_x = measurements.center_of_mass(marked_region)
                    # Convertimos a enteros para coordenadas de píxel
                    puntos_señalados.append((int(round(center_x)), int(round(center_y)))) # Usar round para mejor precisión
                except ValueError as e:
                    print(f"Advertencia: Error calculando centroide para capa {i}: {e}")
                    # Puedes optar por saltar esta capa o registrar el error

    # Formatear la salida
    if puntos_señalados:
        texto_puntos = "Puntos señalados:\n"
        # Eliminar duplicados si múltiples capas marcan el mismo punto aproximadamente
        puntos_unicos = sorted(list(set(puntos_señalados)))
        for i, punto in enumerate(puntos_unicos):
            texto_puntos += f"Punto {i+1}: ({punto[0]}, {punto[1]})\n"
        return texto_puntos
    else:
        return "No se señalaron puntos."

# Interfaz Gradio (sin cambios, ya que el problema estaba en la función)
with gr.Blocks() as demo:
    gr.Markdown("# Señalar Puntos en Imagen con Gradio")
    gr.Markdown("1. Sube una imagen.\n2. Selecciona la herramienta de **pincel** (suele ser la predeterminada).\n3. Haz un **pequeño toque** o clic en la imagen para marcar cada punto. Cada toque se considerará un punto independiente.")

    with gr.Row():
        # Eliminados tool, brush_color, brush_size
        image_input = gr.ImageEditor(
            type="pil",
            label="Sube una imagen y usa el pincel para marcar puntos"
        )

        output_text = gr.Textbox(label="Coordenadas de los Puntos Señalados", lines=10)

    image_input.change(
        fn=procesar_imagen_marcada,
        inputs=image_input,
        outputs=output_text
    )

# Unindent this line to be outside the with block
demo.launch(debug=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

# --- 1. Definir una CNN simple ---
# Vamos a crear un modelo muy básico, solo para demostrar la extracción de filtros.
# Normalmente, entrenarías esto con un dataset para que los filtros aprendan algo significativo.

def build_simple_cnn(input_shape=(32, 32, 3), num_filters=16, kernel_size=(3, 3)):
    model = keras.Sequential([
        # Primera capa convolucional
        layers.Conv2D(num_filters, kernel_size, activation='relu', input_shape=input_shape, padding='same', name='conv1'),
        layers.MaxPooling2D((2, 2)),
        # Puedes añadir más capas si quieres, pero para visualizar patrones simples,
        # la primera capa es la más relevante.
        layers.Flatten(),
        layers.Dense(10, activation='softmax') # Capa densa de salida (ejemplo para clasificación)
    ])
    return model

# --- 2. Crear y "entrenar" un modelo (aquí solo inicializamos para ver los filtros) ---
# Para que los filtros tengan "patrones" aprendidos, la CNN debería ser entrenada
# con un dataset real (ej. CIFAR-10, MNIST).
# Aquí, solo inicializaremos la CNN para poder acceder a los pesos de los filtros.
# Si no la entrenas, los patrones serán aleatorios.

# Puedes cambiar el input_shape según tus imágenes
input_shape = (64, 64, 3) # Ejemplo: imágenes de 64x64 píxeles a color (RGB)
cnn_model = build_simple_cnn(input_shape=input_shape)
cnn_model.summary()

# --- 3. Extraer y Imprimir los Patrones (Filtros) ---
# Los filtros son los pesos de la capa convolucional.
# Acceder a la primera capa convolucional por su nombre
conv1_layer = cnn_model.get_layer('conv1')

# Obtener los pesos (kernels) de la capa convolucional
# weights[0] son los kernels (filtros) y weights[1] son los biases
filters = conv1_layer.get_weights()[0]

print(f"\nForma de los filtros de la capa 'conv1': {filters.shape}")
print(f"La forma es (alto_filtro, ancho_filtro, canales_entrada, numero_filtros)")

# --- 4. Visualizar los Patrones (Filtros) ---
# Cada filtro es un patrón 3D (alto, ancho, canales_entrada)
# Vamos a visualizarlos como imágenes.

num_filters = filters.shape[-1] # Número de filtros en la capa
kernel_height, kernel_width, _, _ = filters.shape

print(f"\nVisualizando los {num_filters} patrones (filtros) de la primera capa:")

fig = plt.figure(figsize=(10, 10))
for i in range(num_filters):
    # Obtener el i-ésimo filtro
    f = filters[:, :, :, i]

    # Normalizar el filtro para la visualización (escalar entre 0 y 1)
    # Esto ayuda a que los patrones sean visibles, ya que los pesos pueden ser negativos.
    f_min, f_max = f.min(), f.max()
    f = (f - f_min) / (f_max - f_min)

    # Crear un subplot para cada filtro
    ax = fig.add_subplot(int(np.ceil(np.sqrt(num_filters))), int(np.ceil(np.sqrt(num_filters))), i + 1)

    # Mostrar el filtro. Si el filtro tiene 3 canales de entrada, lo mostramos como RGB.
    # Si tiene 1 canal (ej. entrada monocromática), lo mostramos en escala de grises.
    if f.shape[-1] == 3: # Si es un filtro RGB
        ax.imshow(f)
    else: # Si es un filtro en escala de grises
        ax.imshow(f[:, :, 0], cmap='gray') # Muestra el primer canal si hay más

    ax.set_xticks([]) # Ocultar ticks del eje x
    ax.set_yticks([]) # Ocultar ticks del eje y
    ax.set_title(f'Filtro {i+1}')

plt.tight_layout()
plt.show()

# --- Notas Importantes ---
# - Si no entrenas la CNN, los filtros serán valores aleatorios, por lo que los "patrones"
#   visualizados también serán aleatorios y no representarán nada significativo.
# - Para ver patrones significativos (bordes, texturas, blobs de color), necesitas
#   **entrenar la CNN con un conjunto de datos relevante**.
# - Los filtros de capas más profundas son más complejos de visualizar e interpretar,
#   ya que operan sobre las características aprendidas por las capas anteriores,
#   no directamente sobre la imagen de entrada.

In [ ]:
# Use shell commands to list files in your directory
# The ! prefix allows running shell commands in Jupyter/Colab
print("Listing files in the main data directory:")
!ls -la "/content/drive/MyDrive/Lunetas/Datasets/Realwomen"

# List files in one of the subdirectories (replace with a real class name)
# This helps you check specific subfolders
print("\nListing files in a sample subdirectory (replace 'ClassName' with a real folder name):")
# Replace 'REPLACE_WITH_A_REAL_CLASS_FOLDER_NAME' with one of the actual folder names listed by the previous command
!ls -la "/content/drive/MyDrive/Lunetas/Datasets/Realwomen/REPLACE_WITH_A_REAL_CLASS_FOLDER_NAME"

# You can use 'find' to search for specific file types or non-standard files
# Example: Find files that don't end with .jpg, .png, .jpeg, .gif, .bmp
# Be careful with this command, it might list directories too.
# A more specific command might be needed depending on what you find.
print("\nSearching for potential non-image files (excluding common image extensions):")
!find "/content/drive/MyDrive/Lunetas/Datasets/Realwomen" -type f ! -iname "*.jpg" ! -iname "*.jpeg" ! -iname "*.png" ! -iname "*.gif" ! -iname "*.bmp"

# Pay close attention to any files listed by the above commands that aren't your images.
# Remove or move any problematic files from the directory in Google Drive.

In [ ]:
# Remove the problematic .webp file
# BE CAREFUL with this command - it will permanently delete the file.
# Make sure the path is correct before running.
print("Attempting to remove the WebP file...")
!rm "/content/drive/MyDrive/Lunetas/Datasets/Realwomen/Undefined/willowsdreaming.webp"
print("File removal attempted. Check the output to confirm.")

# Optional: Verify that the file is gone
print("\nListing files in the 'Undefined' subdirectory after removal attempt:")
!ls -la "/content/drive/MyDrive/Lunetas/Datasets/Realwomen/Undefined"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
import os # Para operaciones con rutas de archivos
import tensorflow as tf # Necesario para image_dataset_from_directory



# --- 1. Definir una CNN simple (Tu código existente) ---
def build_simple_cnn(input_shape, num_filters=16, kernel_size=(3, 3)):
    model = keras.Sequential([
        layers.Conv2D(num_filters, kernel_size, activation='relu', input_shape=input_shape, padding='same', name='conv1'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(num_classes, activation='softmax') # num_classes se definirá más abajo
    ])
    return model

# --- 2. Cargar tus imágenes desde Google Drive ---
# Ruta a tu carpeta principal de datos
data_dir = '/content/Realwomen'

# Definir el tamaño de las imágenes al cargarlas y el tamaño del lote
IMG_HEIGHT = 128 # Puedes ajustar esto según la resolución de tus imágenes
IMG_WIDTH = 128
BATCH_SIZE = 5

# Cargar el dataset usando image_dataset_from_directory
# Esto automáticamente inferirá las clases de las subcarpetas
print(f"Cargando imágenes desde: {data_dir}")
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='int', # Etiquetas como enteros (0, 1, 2, ...)
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest', # Cómo redimensionar si es necesario
    batch_size=BATCH_SIZE,
    shuffle=True # Mezclar los datos
)

# Obtener los nombres de las clases (nombres de las subcarpetas)
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Clases encontradas: {class_names}")
print(f"Número de clases: {num_classes}")

# Ajustar el input_shape de la CNN según el tamaño de las imágenes cargadas
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3) # Asumimos imágenes RGB

# Crear el modelo con el input_shape correcto y el número de clases
cnn_model = build_simple_cnn(input_shape=input_shape, num_filters=16)
cnn_model.summary()

# --- 3. Entrenar el modelo (para que los filtros aprendan patrones) ---
# Es crucial entrenar la CNN para que los filtros aprendan patrones significativos.
# Aquí, compilamos y entrenamos con un pequeño número de épocas como ejemplo.
# En un escenario real, necesitarías dividir tus datos en entrenamiento/validación.

cnn_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False), # Usa True si no tienes softmax en la última capa
    metrics=['accuracy']
)

print("\nEntrenando la CNN (esto tomará un tiempo)...")
# Puedes ajustar el número de epochs
history = cnn_model.fit(
    train_ds,
    epochs=25 # Un número pequeño de épocas para la demostración
)
print("¡Entrenamiento completado!")

# --- 4. Extraer y Visualizar los Patrones (Filtros) después del entrenamiento ---
# Los filtros ahora deberían contener patrones aprendidos.
conv1_layer = cnn_model.get_layer('conv1')
filters = conv1_layer.get_weights()[0]

print(f"\nForma de los filtros de la capa 'conv1': {filters.shape}")
print(f"La forma es (alto_filtro, ancho_filtro, canales_entrada, numero_filtros)")

num_filters = filters.shape[-1]
kernel_height, kernel_width, _, _ = filters.shape

print(f"\nVisualizando los {num_filters} patrones (filtros) aprendidos de la primera capa:")

fig = plt.figure(figsize=(10, 10))
for i in range(num_filters):
    f = filters[:, :, :, i]

    # Normalizar el filtro para la visualización
    f_min, f_max = f.min(), f.max()
    # Evitar división por cero si f_max y f_min son iguales (filtro plano)
    if (f_max - f_min) == 0:
        f = np.zeros_like(f) # Si es plano, solo muestra negro
    else:
        f = (f - f_min) / (f_max - f_min)

    ax = fig.add_subplot(int(np.ceil(np.sqrt(num_filters))), int(np.ceil(np.sqrt(num_filters))), i + 1)

    if f.shape[-1] == 3:
        ax.imshow(f)
    else:
        ax.imshow(f[:, :, 0], cmap='gray')

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Filtro {i+1}')

plt.tight_layout()
plt.show()

# ---
# Puedes ver los primeros elementos del dataset
# Remove the indentation from these lines
for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 10))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
    plt.show() # Move this plt.show() outside the inner loop to show the 3x3 grid at once

In [ ]:
# --- 4. Guardar el modelo ---
model_save_path = '/content/my_trained_cnn_model.keras' # Guarda como archivo .keras

try:
    cnn_model.save(model_save_path)
    print(f"\nModelo guardado exitosamente en: {model_save_path}")
except Exception as e:
    print(f"\nError al guardar el modelo: {e}")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from PIL import Image
import numpy as np

# 1. Cargar el modelo entrenado
model_path = '/content/my_trained_cnn_model.keras'  # Reemplaza con la ruta correcta si es diferente
try:
    loaded_model = keras.models.load_model(model_path)
    print(f"Modelo cargado exitosamente desde: {model_path}")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    exit()  # Detener si no se puede cargar el modelo

# 2. Cargar y preprocesar la imagen que quieres clasificar
image_path = '/Content/test.png'  # Reemplaza con la ruta de tu imagen
try:
    img = Image.open(image_path).resize((128, 128))  # Ajusta el tamaño si es diferente
    img_array = keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Añadir una dimensión de lote (batch)
    img_array = img_array / 255.0  # Normalizar (si tu modelo fue entrenado con imágenes normalizadas)
except Exception as e:
    print(f"Error al cargar o preprocesar la imagen: {e}")
    exit()

# 3. Hacer la predicción
predictions = loaded_model.predict(img_array)
predicted_class = np.argmax(predictions[0])  # Obtener el índice de la clase con mayor probabilidad

# Si tienes los nombres de las clases (class_names) de tu dataset de entrenamiento:
# class_names = ['@EvaJadexo', '@KikoBaka', ...]  # Reemplaza con tus nombres de clase
# predicted_class_name = class_names[predicted_class]
# print(f"La imagen se clasifica como: {predicted_class_name}")
# else:

print(f"La imagen se clasifica como la clase con índice: {predicted_class}")
print(f"Predicciones: {predictions}") # Muestra las probabilidades para cada clase

In [ ]:
import tensorflow as tf
from tensorflow import keras
from PIL import Image
import numpy as np
import os # Asegúrate de que esta librería esté importada

# 1. Cargar el modelo entrenado (resto del código igual)
model_path = '/content/my_trained_cnn_model.keras'
try:
    loaded_model = keras.models.load_model(model_path)
    print(f"Modelo cargado exitosamente desde: {model_path}")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    exit()

# 2. Cargar y preprocesar la imagen que quieres clasificar
# ¡AQUÍ ESTÁ EL CAMBIO IMPORTANTE!
image_path = '/content/test.jpg'  # <--- Cambia .png por .jpg aquí
if not os.path.exists(image_path):
    print(f"Error: La imagen '{image_path}' no se encuentra.")
    exit()

try:
    img = Image.open(image_path).resize((128, 128))
    img_array = keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
except Exception as e:
    print(f"Error al cargar o preprocesar la imagen: {e}")
    exit()

# 3. Hacer la predicción (resto del código igual)
predictions = loaded_model.predict(img_array)
predicted_class = np.argmax(predictions[0])

print(f"La imagen se clasifica como la clase con índice: {predicted_class}")
print(f"Predicciones: {predictions}")

In [ ]:
import os
from PIL import Image
import tensorflow as tf # Aunque solo usaremos PIL aquí, es bueno tenerlo

# Ruta a tu carpeta principal de datos
data_dir = '/content/Realwomen'

# Lista para almacenar los archivos problemáticos
problematic_files = []

print(f"Escaneando archivos en: {data_dir} para detectar formatos inválidos o corruptos...")

# Recorrer todas las subcarpetas y archivos
for root, _, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            # Intentar abrir la imagen con PIL
            # Poner 'Image.MAX_IMAGE_PIXELS = None' si tienes imágenes muy grandes
            Image.open(file_path).verify() # .verify() intenta cargar y verificar el archivo
            # Si llega aquí, la imagen es probablemente válida.
            # También puedes cargarla completamente para una verificación más estricta:
            # img = Image.open(file_path)
            # img.load()
        except (IOError, SyntaxError, Image.UnidentifiedImageError) as e:
            # Capturar errores relacionados con el formato o la corrupción
            print(f"!!! PROBLEMA con el archivo: {file_path} - Error: {e}")
            problematic_files.append(file_path)
        except Exception as e:
            # Capturar cualquier otro error inesperado
            print(f"!!! ERROR INESPERADO al procesar: {file_path} - Error: {e}")
            problematic_files.append(file_path)

print("\n--- ESCANEO COMPLETADO ---")
if problematic_files:
    print(f"Se encontraron {len(problematic_files)} archivo(s) problemático(s):")
    for p_file in problematic_files:
        print(p_file)
    print("\nPor favor, revisa/elimina estos archivos de tu Google Drive y vuelve a ejecutar el código.")
else:
    print("¡No se encontraron archivos problemáticos! Tus imágenes deberían cargarse correctamente.")

In [ ]:
import os
from PIL import Image

# Ruta a tu carpeta principal de datos
data_dir = '/content/Realwomen_RAM' # Asegúrate de que esta sea la carpeta donde están tus datos

print(f"Limpiando y filtrando archivos en: {data_dir}. Solo se conservarán PNG válidos...")

# Recorrer todas las subcarpetas y archivos
for root, _, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)
        file_name, file_ext = os.path.splitext(file)

        # Convertir la extensión a minúsculas para una comparación consistente
        normalized_ext = file_ext.lower().lstrip('.')

        # --- Paso 1: Eliminar archivos que no son PNG ---
        if normalized_ext != 'png':
            try:
                os.remove(file_path)
                print(f"  Eliminado por extensión inválida: {file_path} (era .{normalized_ext})")
            except OSError as e:
                print(f"  Error al eliminar el archivo {file_path}: {e}")
            continue # Pasa al siguiente archivo

        # --- Paso 2: Verificar y eliminar PNGs corruptos ---
        # Si llegamos aquí, la extensión es 'png'. Ahora verificamos si es una imagen válida.
        try:
            with Image.open(file_path) as img:
                img.verify() # Intenta verificar si el archivo es un PNG válido
                # img.load() # Opcional: cargar la imagen completamente para una verificación más estricta
            # Si no hay error, es un PNG válido, lo conservamos.
            # print(f"  Conservado PNG válido: {file_path}") # Descomenta para ver los conservados

        except (IOError, SyntaxError, Image.UnidentifiedImageError) as e:
            # Capturar errores relacionados con el formato o la corrupción
            print(f"!!! Eliminado PNG corrupto: {file_path} - Error: {e}")
            try:
                os.remove(file_path)
            except OSError as e_remove:
                print(f"  Error al eliminar el archivo corrupto {file_path}: {e_remove}")
        except Exception as e:
            # Capturar cualquier otro error inesperado
            print(f"!!! ERROR INESPERADO al procesar {file_path}. Eliminando - Error: {e}")
            try:
                os.remove(file_path)
            except OSError as e_remove:
                print(f"  Error al eliminar el archivo {file_path}: {e_remove}")

print("\n--- PROCESO DE LIMPIEZA COMPLETADO ---")
print("Ahora tu carpeta solo debería contener imágenes PNG válidas.")

In [ ]:
import os
import shutil
from google.colab import drive

# Montar Google Drive
print("Montando Google Drive...")
drive.mount('/content/drive')
print("Google Drive montado.")

# Ruta de la carpeta original en Google Drive
source_dir = '/content/drive/MyDrive/Snsjs/Snj'#'/content/drive/MyDrive/Lunetas/Datasets/Realwomen'
# Ruta de destino en la RAM de Colab
destination_dir = '/content/hubo'#'/content/Realwomen_RAM'

print(f"\nCopiando '{source_dir}' a '{destination_dir}' (en RAM)...")
# Copiar la carpeta recursivamente
shutil.copytree(source_dir, destination_dir)
print("Copia completada.")

# Verificar que la carpeta fue copiada
print(f"Contenido de '{destination_dir}':")
# Puedes listar solo algunas subcarpetas para verificar
for root, dirs, files in os.walk(destination_dir):
    print(f"  Carpeta: {root}")
    # Para no imprimir todos los archivos, solo los primeros 3 si hay muchos
    if files:
        print(f"    Archivos (ejemplo): {files[:3]}...")
    break # Solo para ver el nivel superior, quita esto para ver más

In [ ]:
import os
from PIL import Image
import tensorflow as tf # Aunque solo usaremos PIL aquí, es bueno tenerlo

# Ruta de la carpeta copiada en RAM
target_dir = '/content/Realwomen'
# Nueva extensión deseada (ej. 'jpg', 'png')
new_extension = 'png'

print(f"\nCambiando la extensión de las imágenes en '{target_dir}' a '.{new_extension}'...")

# Recorrer todas las subcarpetas y archivos
for root, _, files in os.walk(target_dir):
    for file in files:
        file_path = os.path.join(root, file)
        file_name, file_ext = os.path.splitext(file)

        # Ignorar archivos que ya tienen la extensión deseada o no son imágenes comunes
        if file_ext.lower().lstrip('.') == new_extension.lower():
            continue

        # Ignorar archivos temporales o de sistema que no sean imágenes
        if file.startswith('.') or file_ext.lower() not in ['.jpg', '.jpeg', '.png', '.gif', '.bmp']:
            print(f"  Saltando archivo no imagen o temporal: {file_path}")
            continue

        try:
            with Image.open(file_path) as img:
                # Crear la nueva ruta con la extensión deseada
                new_file_path = os.path.join(root, f"{file_name}.{new_extension}")

                # Convertir a RGB antes de guardar si la imagen es RGBA (con canal alfa)
                # Esto es útil si vas a guardar como JPG, que no soporta alfa.
                if img.mode == 'RGBA':
                    img = img.convert('RGB')

                img.save(new_file_path)

                # Si la extensión original es diferente, borrar el archivo antiguo
                if file_ext.lower().lstrip('.') != new_extension.lower():
                    os.remove(file_path)
                    # print(f"  Convertido y renombrado: {file_path} -> {new_file_path}")
                # else:
                    # print(f"  Guardado con la misma extensión: {new_file_path}")

        except Exception as e:
            print(f"  Error al procesar el archivo {file_path}: {e}")

print(f"Proceso de cambio de extensión completado. Todas las imágenes deberían ser '.{new_extension}'.")

# Opcional: Volver a verificar la carga del dataset para confirmar
# Este bloque debe estar fuera del bucle de os.walk
data_dir_ram = '/content/Realwomen_RAM'
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32

print(f"\nVerificando la carga de imágenes desde la RAM: {data_dir_ram}")
train_ds_ram = tf.keras.utils.image_dataset_from_directory(
    data_dir_ram,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=True
)
print(f"Clases encontradas en RAM: {train_ds_ram.class_names}")

## Activación seno en imágenes

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
import io

# Define the sine activation function
# It takes a tensor as input and applies tf.sin element-wise.
def sin_activation(x):
    return tf.sin(x)

# --- 0. Prepara una imagen de ejemplo (si no tienes una en /content/) ---
# Vamos a crear una imagen simple de un degradado para que el ejemplo sea autocontenido.
# Puedes reemplazar esto con tu propia imagen PNG.
def create_example_image(filename="/content/405533863d0362e7ac57.png", size=(64, 64)):
    img_array = np.zeros((*size, 3), dtype=np.uint8)
    for y in range(size[0]):
        for x in range(size[1]):
            img_array[y, x, 0] = int(255 * (x / size[1])) # Rojo gradiente horizontal
            img_array[y, x, 1] = int(255 * (y / size[0])) # Verde gradiente vertical
            img_array[y, x, 2] = int(255 * (1 - (x / size[1] + y / size[0]) / 2)) # Azul inverso

    img = Image.fromarray(img_array)
    img.save(filename)
    return filename

#IMG_PATH = create_example_image("/content/405533863d0362e7ac57.png", size=(128, 128))
# Si quieres usar tu propia imagen, descomenta la siguiente línea y pon tu ruta:
IMG_PATH = '/content/descarga.jpeg'

print(f"Usando la imagen: {IMG_PATH}")
if not os.path.exists(IMG_PATH):
    print(f"Error: La imagen '{IMG_PATH}' no se encuentra. Asegúrate de que exista la ruta.")
    exit()

# --- 1. Cargar y Preprocesar la Imagen ---
print("Cargando y preprocesando la imagen...")
original_image = Image.open(IMG_PATH).convert('RGB') # Asegura RGB
width, height = original_image.size
image_pixels = np.array(original_image) / 255.0 # Normaliza los píxeles a [0, 1]

# Visualizar la imagen original
plt.figure(figsize=(4, 4))
plt.imshow(original_image)
plt.title("Imagen Original")
plt.axis('off')
plt.show()

# Crear las coordenadas (x, y) normalizadas a [-1, 1]
# Esto es crucial para que la red funcione bien con coordenadas continuas
x_coords = np.linspace(-1, 1, width)
y_coords = np.linspace(-1, 1, height)
# Crear una malla de coordenadas
X, Y = np.meshgrid(x_coords, y_coords)

# Aplanar las coordenadas y los píxeles para el entrenamiento
# `input_coords` tendrá la forma (num_pixels, 2) -> [[x1, y1], [x2, y2], ...]
# `output_colors` tendrá la forma (num_pixels, 3) -> [[R1, G1, B1], [R2, G2, B2], ...]
input_coords = np.stack([X.flatten(), Y.flatten()], axis=-1)
output_colors = image_pixels.reshape(-1, 3)

print(f"Imagen de {width}x{height} píxeles.")
print(f"Preparados {len(input_coords)} pares (coordenada, color) para el entrenamiento.")

# --- 2. Definir la Red Neuronal (la "Función Matemática") ---
# Una MLP simple con activaciones sinusoidales (SIREN-like) suele funcionar mejor para INR
# ya que pueden aprender señales de alta frecuencia. Si no, ReLU o LeakyReLU también funcionan.
print("Construyendo la red neuronal (función implícita)...")
model = keras.Sequential([
    keras.layers.Input(shape=(2,)), # Entrada: (x, y) coordenadas
    keras.layers.Dense(256, activation=sin_activation), # Usar la función definida
    keras.layers.Dense(256, activation=sin_activation), # Usar la función definida
    keras.layers.Dense(256, activation=sin_activation), # Usar la función definida
    keras.layers.Dense(256, activation=sin_activation), # Usar la función definida
    keras.layers.Dense(3, activation='sigmoid') # Salida: (R, G, B) colores en [0, 1]
])

# --- 3. Compilar y Entrenar la Red ---
print("Compilando y entrenando la red neuronal...")
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='mse') # Mean Squared Error: queremos que prediga los colores con precisión

# El entrenamiento es el proceso de "convertir" el PNG en la función
history = model.fit(input_coords, output_colors,
                    epochs=100, # Unas 100-200 épocas suelen ser un buen comienzo para imágenes pequeñas
                    batch_size=256,
                    verbose=0) # No mostrar cada época, solo resumen
print(f"Entrenamiento completado en {len(history.history['loss'])} épocas. Pérdida final: {history.history['loss'][-1]:.4f}")

# --- 4. Renderizar la Imagen a Partir de la Función Entrenada ---
# Ahora, la red es nuestra "función matemática". Podemos usarla para generar la imagen.
print("\nGenerando la imagen a partir de la función entrenada...")

# Puedes especificar la resolución que quieras, incluso mayor que la original
render_width = 256
render_height = 256

# Crear las coordenadas para la nueva resolución
render_x_coords = np.linspace(-1, 1, render_width)
render_y_coords = np.linspace(-1, 1, render_height)
Render_X, Render_Y = np.meshgrid(render_x_coords, render_y_coords)
render_input_coords = np.stack([Render_X.flatten(), Render_Y.flatten()], axis=-1)

# Usar la red entrenada para predecir los colores para estas nuevas coordenadas
# Esto es como "dibujar" el PDF a una resolución específica
predicted_colors = model.predict(render_input_coords, verbose=0)

# Reshape los colores de vuelta a la forma de una imagen
rendered_image_array = predicted_colors.reshape((render_height, render_width, 3))

# Visualizar la imagen generada por la función
plt.figure(figsize=(4, 4))
plt.imshow(np.clip(rendered_image_array, 0, 1)) # Asegura que los valores estén en [0, 1] para imshow
plt.title(f"Imagen Generada por la Función (Resolución: {render_width}x{render_height})")
plt.axis('off')
plt.show()

# --- Opcional: Guardar la "Función Matemática" (el modelo de Keras) ---
# Puedes guardar el modelo y cargarlo más tarde para generar la imagen.
model_save_path = '/content/image_function_model.keras'
model.save(model_save_path)
print(f"\nLa 'función matemática' (modelo de Keras) guardada en: {model_save_path}")

# Ejemplo de cómo cargarla y usarla (descomenta para probar)
# loaded_model = keras.models.load_model(model_save_path, compile=False, custom_objects={'sin_activation': sin_activation}) # Necesitas pasar custom_objects para la activación custom
# loaded_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse') # Recompilar
# loaded_predicted_colors = loaded_model.predict(render_input_coords)
# loaded_rendered_image_array = loaded_predicted_colors.reshape((render_height, render_width, 3))
# plt.imshow(loaded_rendered_image_array)
# plt.title("Cargado y Renderizado")
# plt.show()

## GAN: generador y discriminador

Es la DCGAN estándar de la industria (Radford et al., 2015) para imágenes de 128×128, sin una idea nueva que medir. Se dejó una versión (había 3).

In [ ]:
# train_gan.py

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

# --- Configuración global ---
config = {
    "data_dir": "/content/drive/MyDrive/Lunetas/L",
    "latent_dim": 100,
    "epochs": 200,
    "image_size": (128, 128),
    "batch_size": 128,
    "model_path_g": "/content/gan_generator_128x128.keras",
    "model_path_d": "/content/gan_discriminator_128x128.keras",
    "lr": 2e-4,
    "beta_1": 0.5,
    "save_freq": 150
}

# --- Dataset ---
def load_dataset():
    if not os.path.exists(config["data_dir"]):
        raise FileNotFoundError(f"No se encuentra la carpeta: {config['data_dir']}")

    dataset = tf.keras.preprocessing.image_dataset_from_directory(
        directory=config["data_dir"],
        label_mode=None,
        image_size=config["image_size"],
        batch_size=config["batch_size"],
        shuffle=True
    ).map(lambda x: (tf.cast(x, tf.float32) / 127.5) - 1.0)

    return dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

# --- Modelos ---
def build_generator():
    model = tf.keras.Sequential(name="generator")
    model.add(tf.keras.layers.Dense(8 * 8 * 512, input_dim=config["latent_dim"], use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.LeakyReLU())
    model.add(tf.keras.layers.Reshape((8, 8, 512)))

    for filters in [256, 128, 64]:
        model.add(tf.keras.layers.Conv2DTranspose(filters, 4, strides=2, padding='same', use_bias=False))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.LeakyReLU())

    model.add(tf.keras.layers.Conv2DTranspose(3, 4, strides=2, padding='same', activation='tanh', use_bias=False))
    return model

def build_discriminator():
    model = tf.keras.Sequential(name="discriminator")
    model.add(tf.keras.layers.Input(shape=(128, 128, 3)))

    for filters in [64, 128, 256, 512]:
        model.add(tf.keras.layers.Conv2D(filters, 4, strides=2, padding='same'))
        if filters != 64:
            model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.LeakyReLU())
        model.add(tf.keras.layers.Dropout(0.3))

    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    return model

# --- Pérdidas y optimizadores ---
cross_entropy = tf.keras.losses.BinaryCrossentropy()

def discriminator_loss(real_output, fake_output):
    return cross_entropy(tf.ones_like(real_output), real_output) + \
           cross_entropy(tf.zeros_like(fake_output), fake_output)

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator = build_generator()
discriminator = build_discriminator()

gen_optimizer = tf.keras.optimizers.Adam(learning_rate=config["lr"], beta_1=config["beta_1"])
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=config["lr"], beta_1=config["beta_1"])

@tf.function
def train_step(images):
    noise = tf.random.normal([images.shape[0], config["latent_dim"]])
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_imgs = generator(noise, training=True)
        real_out = discriminator(images, training=True)
        fake_out = discriminator(gen_imgs, training=True)
        gen_loss = generator_loss(fake_out)
        disc_loss = discriminator_loss(real_out, fake_out)

    gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return gen_loss, disc_loss

# --- Entrenamiento ---
seed = tf.random.normal([16, config["latent_dim"]])

def generate_and_save_images(model, epoch, test_input):
    preds = model(test_input, training=False)
    preds = (preds * 0.5) + 0.5  # [-1, 1] -> [0, 1]
    fig = plt.figure(figsize=(4, 4))
    for i in range(preds.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow(np.clip(preds[i].numpy(), 0, 1))
        plt.axis('off')
    plt.savefig(f'/content/image_at_epoch_{epoch:04d}.png')
    plt.close(fig)

def train():
    dataset = load_dataset()
    for epoch in range(config["epochs"]):
        gen_losses, disc_losses = [], []
        for image_batch in tqdm(dataset, desc=f"Epoch {epoch+1}/{config['epochs']}"):
            g_loss, d_loss = train_step(image_batch)
            gen_losses.append(g_loss.numpy())
            disc_losses.append(d_loss.numpy())

        print(f"\nEpoch {epoch+1} | Gen Loss: {np.mean(gen_losses):.4f} | Disc Loss: {np.mean(disc_losses):.4f}")

        if (epoch + 1) % config["save_freq"] == 0 or (epoch + 1) == config["epochs"]:
            generate_and_save_images(generator, epoch + 1, seed)

    generator.save(config["model_path_g"])
    discriminator.save(config["model_path_d"])
    print("\nModelos guardados exitosamente.")

if __name__ == "__main__":
    train()

## Preparar el video: fotogramas en `.npy`

Extrae fotogramas de un video (o de una carpeta de imágenes), los mejora y los guarda como `.npy` para las tripletas.

In [ ]:
# ======================== CONFIGURACIÓN GENERAL ========================

CONFIG = {
    # 📁 Entrada
    "usar_video": True,  # True: usa video. False: usa carpeta de imágenes.
    "ruta_video": "/content/951186ff829629567f3130b8974fa307_1749686021858.mp4",
    "carpeta_imagenes": "/content/imagenes/",

    # 📤 Salida
    "carpeta_salida": "/content/npy_frames/",
    "guardar_npy": True,
    "guardar_jpg_mejorado": True,

    # ⚙️ Mejora visual
    "nitidez_factor": 1.5,         # >1.0 = más nítido
    "denoise_sigmaColor": 50,      # Suaviza ruido de color
    "denoise_sigmaSpace": 50,      # Suaviza detalles cercanos

    # 🧠 Control
    "conservar_canales_extra": True,  # True = permite RGBA o más
    "extraer_cada_N_frames": 24,       # 1 = todos los frames, 2 = cada 2 frames...
    "mostrar_info": True,
}

In [ ]:
# ======================== PREPROCESAMIENTO ========================

import os, glob, cv2, numpy as np
import imageio.v3 as iio
from PIL import Image, ImageEnhance
from tqdm import tqdm

os.makedirs(CONFIG["carpeta_salida"], exist_ok=True)

def mejorar_imagen(img):
    pil = Image.fromarray(img[:, :, :3])
    sharp = ImageEnhance.Sharpness(pil).enhance(CONFIG["nitidez_factor"])
    img_sharp = np.array(sharp)

    if img.shape[2] > 3:
        img_sharp = np.concatenate([img_sharp, img[:, :, 3:]], axis=2)

    mejorada = cv2.bilateralFilter(
        img_sharp,
        d=9,
        sigmaColor=CONFIG["denoise_sigmaColor"],
        sigmaSpace=CONFIG["denoise_sigmaSpace"]
    )

    return mejorada

def guardar(nombre, img):
    normalizada = img.astype(np.float32) / 255.0
    if CONFIG["guardar_npy"]:
        np.save(os.path.join(CONFIG["carpeta_salida"], f"{nombre}.npy"), normalizada)
    if CONFIG["guardar_jpg_mejorado"]:
        iio.imwrite(os.path.join(CONFIG["carpeta_salida"], f"{nombre}.jpg"),
                    (normalizada * 255).astype(np.uint8))

def procesar_imagen(img, index):
    if not CONFIG["conservar_canales_extra"] and img.shape[2] > 3:
        img = img[:, :, :3]
    mejorada = mejorar_imagen(img)
    guardar(f"frame_{index:05}", mejorada)
    if CONFIG["mostrar_info"]:
        print(f"Frame {index} - shape: {mejorada.shape} dtype: {mejorada.dtype}")

In [ ]:
# ======================== MODO VIDEO ========================

def extraer_desde_video():
    cap = cv2.VideoCapture(CONFIG["ruta_video"])
    index = 0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    for i in tqdm(range(total), desc="Extrayendo video"):
        ret, frame = cap.read()
        if not ret: break
        if i % CONFIG["extraer_cada_N_frames"] != 0: continue

        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        procesar_imagen(img, index)
        index += 1

    cap.release()

In [ ]:
# ======================== EJECUCIÓN ========================

if CONFIG["usar_video"]:
    extraer_desde_video()
else:
    extraer_desde_carpeta()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image # Para guardar como JPG

# Define la ruta al archivo .npy
npy_file_path = "/content/npy_frames/frame_00000.npy"
output_jpg_path = "/content/frame_00000.jpg" # Ruta donde se guardará la imagen JPG

try:
    # Cargar el array NumPy desde el archivo .npy
    image_array = np.load(npy_file_path)

    # El array .npy debe estar normalizado entre 0 y 1.
    # Para convertirlo a imagen (0-255), multiplicamos por 255 y lo convertimos a tipo de dato uint8.
    # Además, si el NPY está en formato (C, H, W) como lo genera PyTorch, necesitamos transponerlo a (H, W, C) para PIL y Matplotlib.

    # Intenta determinar el formato (C, H, W) o (H, W, C)
    if image_array.ndim == 3 and image_array.shape[0] in [1, 3, 4]: # Podría ser (canales, alto, ancho)
        # Si el primer dimension es pequeño (1, 3 o 4) y el segundo es grande, asumimos (C, H, W)
        if image_array.shape[1] > image_array.shape[0]:
            image_array = np.transpose(image_array, (1, 2, 0)) # Transponer a (H, W, C)
            print("Transpuesto de (C, H, W) a (H, W, C)")

    # Asegurarse de que el rango es [0, 1] antes de convertir a 0-255
    # Esto es crucial si el .npy está normalizado pero no estrictamente entre 0 y 1
    image_array = np.clip(image_array, 0.0, 1.0)
    image_255 = (image_array * 255).astype(np.uint8)

    # Si es una imagen en escala de grises, eliminar la dimensión extra del canal si existe (ej. (H, W, 1) a (H, W))
    if image_255.ndim == 3 and image_255.shape[2] == 1:
        image_255 = image_255.squeeze(axis=2)

    # Guardar la imagen como JPG
    # PIL.Image.fromarray maneja (H, W) para escala de grises y (H, W, C) para color.
    img = Image.fromarray(image_255)
    img.save(output_jpg_path)
    print(f"Imagen guardada exitosamente como {output_jpg_path}")

    # Visualizar la imagen (opcional)
    plt.imshow(img)
    plt.title(f"Imagen generada desde {npy_file_path}")
    plt.axis('off')
    plt.show()

except FileNotFoundError:
    print(f"Error: El archivo {npy_file_path} no fue encontrado. Por favor, verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error al procesar el archivo NPY: {e}")

## FrameInterpolator: predecir el fotograma siguiente

LSTM sobre la imagen aplanada que, a partir de un fotograma, dibuja el siguiente. Con secuencias de largo 1 la LSTM no recuerda nada (es una red densa grande). Medido en el banco (`laboratorio/banco/video.py`, resultados en `laboratorio/banco/resultados-04-video.md`).

In [ ]:
# frame_interpolator_colab.py
import torch
import torch.nn as nn
import torch.optim as optim
import imageio
import cv2
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

# ---------------- CONFIGURACION ---------------- #
config = {
    "video_path": '/content/951186ff829629567f3130b8974fa307_1749686021858.mp4',
    "resize": (160, 90),
    "max_episodes": 50,
    "max_frames": 3000,
    "lr": 1e-4,
    "hidden_size": 1024,
    "num_layers": 2,
}

width, height = config["resize"]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

if use_amp:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    torch.backends.cudnn.benchmark = True

print(f"Usando dispositivo: {device}")
if use_amp:
    print("AMP habilitado")


# ---------------- VIDEO DATASET ---------------- #
class VideoDataset(torch.utils.data.Dataset):
    def __init__(self, frames):
        self.frames = frames

    def __len__(self):
        return len(self.frames) - 1

    def __getitem__(self, idx):
        return self.frames[idx], self.frames[idx + 1]


def cargar_video(path, max_frames):
    frames = []
    reader = imageio.get_reader(path, 'ffmpeg')
    for i, frame in enumerate(reader):
        if i >= max_frames:
            break
        if frame.ndim == 3 and frame.shape[2] == 3:
            frame = cv2.resize(frame, (width, height))
            tensor = torch.tensor(frame, dtype=torch.float32) / 255.0
            tensor = tensor.permute(2, 0, 1)
            frames.append(tensor)
    reader.close()
    return torch.stack(frames) if frames else None


# ---------------- MODELO ---------------- #
class FrameInterpolator(nn.Module):
    def __init__(self, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size=height * width * 3, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, height * width * 3)

    def forward(self, x):
        B = x.size(0)
        x = x.view(B, -1).unsqueeze(1)
        out, (h_n, _) = self.lstm(x)
        out = self.fc(h_n[-1])
        return torch.clamp(out.view(B, 3, height, width), 0, 1)


# ---------------- METRICAS ---------------- #
def psnr(predicted, target):
    mse = torch.mean((predicted - target) ** 2).item()
    return 100 if mse == 0 else 20 * np.log10(1.0 / np.sqrt(mse))


# ---------------- ENTRENAMIENTO ---------------- #
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_psnr = 0, 0
    pbar = tqdm(loader, desc="Entrenando")

    for input_frame, target_frame in pbar:
        input_frame = input_frame.to(device)
        target_frame = target_frame.to(device)
        optimizer.zero_grad()

        if use_amp:
            with autocast():
                pred = model(input_frame)
                loss = criterion(pred, target_frame)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(input_frame)
            loss = criterion(pred, target_frame)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        total_psnr += psnr(pred.detach().cpu(), target_frame.cpu())
        pbar.set_postfix({"loss": loss.item(), "psnr": total_psnr / (pbar.n + 1)})

    return total_loss / len(loader), total_psnr / len(loader)


# ---------------- MAIN ---------------- #
if __name__ == "__main__":
    video = cargar_video(config["video_path"], config["max_frames"])
    if video is None:
        raise RuntimeError("No se pudo cargar el video.")

    dataset = VideoDataset(video)
    loader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True,
                                         num_workers=os.cpu_count())

    model = FrameInterpolator(config["hidden_size"], config["num_layers"]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.MSELoss()

    best_psnr = -1
    for epoch in range(config["max_episodes"]):
        loss, epoch_psnr = train(model, loader, optimizer, criterion)
        print(f"Epoch {epoch+1}: Loss={loss:.4f}, PSNR={epoch_psnr:.2f}dB")
        if epoch_psnr > best_psnr:
            best_psnr = epoch_psnr
            torch.save(model.state_dict(), "frame_interpolator_best.pth")
            print("Mejor modelo guardado.")

    torch.save(model.state_dict(), "frame_interpolator_final.pth")
    print("Modelo final guardado.")

In [ ]:


# Pass the required arguments from the config dictionary to the FrameInterpolator constructor
model = FrameInterpolator(hidden_size=config["hidden_size"], num_layers=config["num_layers"]).to(device)
model.load_state_dict(torch.load('/content/frame_interpolator_final.pth', map_location=device, weights_only=True))
model.eval()

# --- CARGAR NUEVO VIDEO (puede ser el mismo) ---
# Ensure video_path is defined, either use the one from config or another path
video_path = config["video_path"] # Using the video path from the config
video_test = cargar_video(video_path, max_frames=20)

if video_test is None or len(video_test) < 2:
    print("No hay suficientes frames para interpolar.")
else:
    with torch.no_grad():
        for i in range(len(video_test) - 1):
            input_frame = video_test[i].unsqueeze(0).to(device)
            target_frame = video_test[i+1]

            predicted_frame = model(input_frame).squeeze(0).cpu()

            # Preparar para visualización
            input_np = input_frame.squeeze(0).cpu().permute(1, 2, 0).numpy()
            predicted_np = predicted_frame.permute(1, 2, 0).numpy()
            target_np = target_frame.permute(1, 2, 0).numpy()

            # Ensure similitud_color is defined or use the psnr function from the previous cell
            # psnr = similitud_color(predicted_frame, target_frame) # Assuming similitud_color exists and works with tensors
            # If similitud_color is not defined, use the psnr function defined earlier
            frame_psnr = psnr(predicted_frame, target_frame.cpu()) # Calculate PSNR for the frame
            print(f"Frame {i+1}: PSNR = {frame_psnr:.2f} dB")

            # Mostrar resultados
            fig, axs = plt.subplots(1, 3, figsize=(15, 5))
            axs[0].imshow(input_np)
            axs[0].set_title("Input Frame (i)")
            axs[1].imshow(predicted_np)
            axs[1].set_title("Predicted Frame (i+1)")
            axs[2].imshow(target_np)
            axs[2].set_title("Real Frame (i+1)")
            for ax in axs: ax.axis("off")
            plt.tight_layout()
            plt.show()

# Prototipos pequeños

## Clasificador de imágenes
Espera las imágenes en `./content/train/<clase>/`. Sin esa carpeta falla con `NotFoundError`.

In [ ]:
import tensorflow as tf

# Cargar el dataset
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "./content/train",
    image_size=(224, 224),
    batch_size=32,
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "./content/test",
    image_size=(224, 224),
    batch_size=32,
)

# Definir la arquitectura de la red
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

# Compilar el modelo
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Entrenar el modelo
model.fit(train_dataset, epochs=10)

# Evaluar el modelo
model.evaluate(test_dataset)

# Guardar el modelo
model.save("modelo.h5")